# Bruise segmentation — FINAL: SegFormer (custom loop) + native YOLO, all 185 test images

One notebook, five models, three seeds each. **SegFormer** trains with its own custom
loop at each model's best batch; **YOLO** trains with **native Ultralytics** and is
scored **two ways**. Fairness across skin tone for everything. Survives session death.

| model | trained by | batch | distill α | evaluated |
|---|---|---|---|---|
| SegFormer-B2 teacher | custom loop | per-model best (~32) | — | sigmoid + val-swept thr |
| SegFormer-B0 direct | custom loop | per-model best (~64) | — | sigmoid + val-swept thr |
| SegFormer-B0 distilled | custom loop | per-model best (~64) | **0.6** | sigmoid + val-swept thr |
| YOLO26n direct | **native Ultralytics** | native auto | — | **(a) argmax  (b) /255 swept** |
| YOLO26n distilled | **native Ultralytics** | native auto | **0.4** | **(a) argmax  (b) /255 swept** |

**Why this split (read `docs/training_notebook_explained.md` for the full story):**
- SegFormer keeps its own recipe — it's the healthy, reproducible pipeline.
- YOLO is trained **natively** (mosaic + EMA + letterbox + its own LR), *not* forced
  onto SegFormer's transformer recipe — because a CNN and a transformer don't share an
  optimal LR, and native YOLO is what reproduces its ~0.83 median.
- YOLO gets **two evaluation paths** so you see both its best case (native argmax) and
  the SegFormer-comparable geometry (custom /255). The miss rate is the honest axis.

**Data:** the zip ships **native-resolution** images. At setup we build a 640-stretch
cache **once** (for SegFormer training + the custom-YOLO path); native YOLO trains
straight off the native images so Ultralytics' letterbox reproduces its result.

## §1 · Configuration

In [ ]:
CFG = dict(
    img_size        = 640,
    zip_name        = "bruise_colab_final.zip",
    drive_dir       = "/content/drive/MyDrive/bruise_segmentation_gpu",
    work_dir        = "/content/bruise_final",          # local SSD, wiped on disconnect

    # ── SegFormer recipe (custom loop, unchanged) ───────────────────────────
    epochs          = 100,
    patience        = 15,
    batch_mode      = "per_model",   # each SegFormer model uses its own largest batch
    effective_batch = 8,             # fallback only; per_model probes up to max_probe_batch
    max_probe_batch = 64,
    vram_target     = 0.75,
    backbone_lr     = 6e-5,
    head_lr         = 6e-4,
    betas           = (0.9, 0.999),
    weight_decay    = 0.01,
    warmup_fraction = 0.01,
    poly_power      = 1.0,
    gradient_clip   = 1.0,
    amp             = True,
    workers         = 4,
    segformer_alpha = 0.6,           # SegFormer distillation weight (project's value)
    aux_weight      = 0.4,

    # ── YOLO recipe (NATIVE Ultralytics, its own settings) ──────────────────
    yolo_alpha      = 0.4,           # offline pseudo-mask KD (below 0.5 -> non-degenerate)
    yolo_batch      = -1,            # -1 = Ultralytics auto-batch (native "largest that fits")
    yolo_optimizer  = "auto",
    yolo_lrf        = 0.01,
    yolo_warmup_epochs = 3,
    yolo_weight_decay  = 0.0005,
    yolo_close_mosaic  = 10,
    pseudo_threshold   = 0.50,

    # ── shared ──────────────────────────────────────────────────────────────
    seeds           = (0, 1, 2),
    cut_min         = -6.0, cut_max = 6.0, cut_steps = 481,   # SegFormer logit-cut sweep
    prob_min        = 0.01, prob_max = 0.99, prob_steps = 197, # YOLO custom-path prob sweep
    drive_sync_every = 2,
)

SEGFORMER_MODELS = {
    "segformer_b2_teacher":   dict(arch="segformer", size="b2", distill=False),
    "segformer_b0_direct":    dict(arch="segformer", size="b0", distill=False),
    "segformer_b0_distilled": dict(arch="segformer", size="b0", distill=True),
}
YOLO_MODELS = {
    "yolo_sem_direct":    dict(distill=False),
    "yolo_sem_distilled": dict(distill=True),
}
print(f"{len(SEGFORMER_MODELS)} SegFormer + {len(YOLO_MODELS)} YOLO models x {len(CFG['seeds'])} seeds")

## §2 · Drive, GPU, dependencies

In [ ]:
import os, sys, time
from pathlib import Path
from google.colab import drive
drive.mount("/content/drive")

import torch
if not torch.cuda.is_available():
    raise RuntimeError("No GPU. Runtime -> Change runtime type -> GPU (A100). Refusing to run on CPU.")
print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
%pip install -q "transformers>=4.40,<6" "ultralytics>=8.4,<9" "albumentations>=2.0,<3" "scipy>=1.11" "pandas>=2.0" "matplotlib>=3.7" "pyyaml"
import transformers, ultralytics, albumentations
print("transformers", transformers.__version__, "| ultralytics", ultralytics.__version__)

## §3 · Unpack (native-res) and build the 640 cache once

The zip is native resolution (~2.7 GB). We unzip to local SSD, then build a
640×640 stretch cache **once** — SegFormer trains on that (fast, no per-epoch
resize) and the custom-YOLO path reads its 640 images from it. Native YOLO uses the
native-res images directly.

In [ ]:
import zipfile, cv2, numpy as np, pandas as pd

ZIP_SRC = Path(CFG["drive_dir"]) / CFG["zip_name"]
WORK    = Path(CFG["work_dir"])
if not ZIP_SRC.exists():
    raise FileNotFoundError(f"{ZIP_SRC} not found. Build with scripts/42 and upload.")
if ZIP_SRC.stat().st_size < 2e9:
    raise RuntimeError(f"{ZIP_SRC} is only {ZIP_SRC.stat().st_size/1e9:.2f} GB; expected ~2.7 GB native-res package.")

if not (WORK / "manifests" / "train.csv").exists():
    WORK.mkdir(parents=True, exist_ok=True)
    t0 = time.time()
    with zipfile.ZipFile(ZIP_SRC) as zf:
        zf.extractall(WORK)
    print(f"unzipped in {time.time()-t0:.0f}s")

MAN = {s: pd.read_csv(WORK / "manifests" / f"{s}.csv") for s in ("train", "val", "test")}
for s, df in MAN.items():
    print(f"{s:>5}: {len(df):>3} images, {df['subject'].nunique():>3} subjects")
for a, b in [("train","val"),("train","test"),("val","test")]:
    assert not (set(MAN[a]["subject"]) & set(MAN[b]["subject"])), f"subject leak {a}/{b}"
    assert not (set(MAN[a]["stem"]) & set(MAN[b]["stem"])), f"image leak {a}/{b}"
assert (len(MAN["train"]),len(MAN["val"]),len(MAN["test"]))==(697,134,185)
print("PASS -- no leakage.")

RUNS_DIR = Path(CFG["drive_dir"]) / "runs_final"
OUT_DIR  = Path(CFG["drive_dir"]) / "results_final"
RUNS_DIR.mkdir(parents=True, exist_ok=True); OUT_DIR.mkdir(parents=True, exist_ok=True)
print("checkpoints ->", RUNS_DIR)

In [ ]:
# Build the 640 stretch cache once (image bilinear, mask nearest -- albumentations'
# defaults, so this is bit-exact to what the training dataloader would compute live).
CACHE640 = WORK / "cache640"
def build_cache(df, split):
    idir = CACHE640 / split / "images"; mdir = CACHE640 / split / "masks"
    idir.mkdir(parents=True, exist_ok=True); mdir.mkdir(parents=True, exist_ok=True)
    for _, r in df.iterrows():
        ip = idir / f"{r.stem}.png"; mp = mdir / f"{r.stem}.png"
        if not ip.exists():
            im = cv2.imread(str(WORK / r.image_path), cv2.IMREAD_COLOR)
            cv2.imwrite(str(ip), cv2.resize(im, (640,640), interpolation=cv2.INTER_LINEAR))
        if not mp.exists():
            m = cv2.imread(str(WORK / r.mask_path), cv2.IMREAD_GRAYSCALE)
            if m.ndim == 3: m = m[..., 0]
            b = (m > 0).astype(np.uint8)
            cv2.imwrite(str(mp), cv2.resize(b, (640,640), interpolation=cv2.INTER_NEAREST) * 255)

if not (CACHE640 / "test" / "images").exists():
    t0 = time.time()
    for s in ("train","val","test"): build_cache(MAN[s], s)
    print(f"640 cache built in {time.time()-t0:.0f}s")
else:
    print("640 cache present")

# Manifests that point at the 640 cache (relative to CACHE640) -- SegFormer + custom-YOLO use these.
MAN640 = {}
for s, df in MAN.items():
    d = df.copy()
    d["image_path"] = d["stem"].apply(lambda x: f"{s}/images/{x}.png")
    d["mask_path"]  = d["stem"].apply(lambda x: f"{s}/masks/{x}.png")
    MAN640[s] = d

## §4 · The library

SegFormer modules are **reused verbatim** from `bruise_colab_train_all.ipynb` (tested
end-to-end). One new module, `yolo_native.py`, wraps native Ultralytics training and
the two YOLO prediction paths.

In [ ]:
%cd /content
from pathlib import Path
Path("/content/bruisekit").mkdir(parents=True, exist_ok=True)
print("package dir ready:", Path("/content/bruisekit").is_dir())

In [ ]:
%%writefile bruisekit/__init__.py
"""Bruise segmentation training kit. See docs/training_notebook_explained.md."""

In [ ]:
%%writefile bruisekit/data.py
"""One dataloader for every model: one disk read, one resize, one augmentation.

THE DATASET EMITS RAW [0,1] PIXELS -- IT DOES NOT NORMALISE
------------------------------------------------------------
SegFormer wants ImageNet-normalised input; YOLO wants plain /255. That is not a
style preference, it is a property of the trained weights: Ultralytics trains on
/255 and its BatchNorms carry frozen running statistics for that distribution.
Feeding YOLO ImageNet-normalised pixels makes it under-fire by 4x and caps it at
Dice 0.479 with NO threshold able to recover it.

So pixel scale belongs to the MODEL, not the loader (see models.py). The loader
emits raw [0,1] and each wrapper applies its own scale. Every model therefore
shares one disk read, one resize and one augmentation -- the geometry is
identical by construction -- while each still sees the distribution it was
trained for. The alternative (normalise here, un-normalise for YOLO) works but
carries a needless roundtrip and invites exactly the bug it is working around.
"""
from __future__ import annotations

from pathlib import Path

import albumentations as A
import cv2
import numpy as np
import pandas as pd
import torch
from albumentations.pytorch import ToTensorV2
from torch.utils.data import Dataset


def build_augmentation(training: bool, img_size: int) -> A.Compose:
    """The augmentation pipeline. Identical for all five models.

    A.Resize is a NO-OP on the packaged 640x640 PNGs -- it is kept as a cheap
    guard so a wrong-sized file fails as a shape error here rather than as a
    silent geometry mismatch 40 minutes into training.

    A.Normalize(mean=0, std=1, max_pixel_value=255) is exactly x/255: albumentations
    computes (img - mean*max_pixel_value) / (std*max_pixel_value).
    """
    to_unit = A.Normalize(mean=(0.0, 0.0, 0.0), std=(1.0, 1.0, 1.0), max_pixel_value=255.0)
    resize = [A.Resize(height=img_size, width=img_size)]
    if not training:
        return A.Compose(resize + [to_unit, ToTensorV2()])
    return A.Compose(resize + [
        A.HorizontalFlip(p=0.5),
        A.VerticalFlip(p=0.3),
        A.RandomBrightnessContrast(brightness_limit=0.20, contrast_limit=0.20, p=0.4),
        A.HueSaturationValue(hue_shift_limit=10, sat_shift_limit=15, val_shift_limit=10, p=0.3),
        A.GaussNoise(p=0.2),
        to_unit, ToTensorV2(),
    ])


class BruiseDataset(Dataset):
    """Reads the 640x640 PNG package. Returns (x[3,H,W] in [0,1], y[1,H,W] in {0,1}, stem)."""

    def __init__(self, df: pd.DataFrame, root: Path, img_size: int, training: bool = False):
        self.df = df.reset_index(drop=True)
        self.root = Path(root)
        self.img_size = img_size
        self.tfm = build_augmentation(training, img_size)

    def __len__(self) -> int:
        return len(self.df)

    def __getitem__(self, idx: int):
        r = self.df.iloc[idx]

        img = cv2.imread(str(self.root / r.image_path), cv2.IMREAD_COLOR)
        if img is None:
            raise RuntimeError(f"Cannot read image: {r.image_path}")
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

        mask = cv2.imread(str(self.root / r.mask_path), cv2.IMREAD_GRAYSCALE)
        if mask is None:
            raise RuntimeError(f"Cannot read mask: {r.mask_path}")
        # IMREAD_GRAYSCALE is documented to return (H, W) but returns (H, W, 1) in any
        # process where ultralytics has been imported -- it monkey-patches cv2.imread,
        # and import ORDER does not help. That trailing axis survives ToTensorV2 and
        # makes y [B,1,H,W,1]; dice(pred[H,W], gt[H,W,1]) then BROADCASTS to [H,W,H]
        # and returns nonsense -- a pixel-perfect prediction scores 63.9 instead of 1.0.
        # Squeezing here is a no-op on an unpatched cv2 and fixes every consumer at once.
        if mask.ndim == 3:
            mask = mask[..., 0]
        mask = (mask > 0).astype(np.float32)

        aug = self.tfm(image=img, mask=mask)
        x = aug["image"].float()
        y = aug["mask"].unsqueeze(0).float()
        assert y.shape == (1, self.img_size, self.img_size), f"bad mask shape {y.shape} for {r.stem}"
        return x, y, str(r.stem)


def make_loader(df, root, img_size, batch_size, training, workers, seed=0):
    """DataLoader with seeded, reproducible shuffling and worker RNG."""
    ds = BruiseDataset(df, root, img_size, training=training)
    gen = torch.Generator()
    gen.manual_seed(seed)

    def _init_worker(worker_id: int) -> None:
        # Each worker gets a distinct but seed-derived RNG stream, so augmentation
        # is reproducible for a given seed AND workers never draw identical noise.
        np.random.seed(seed * 1000 + worker_id)

    return torch.utils.data.DataLoader(
        ds, batch_size=batch_size, shuffle=training, drop_last=training,
        num_workers=workers, pin_memory=True,
        persistent_workers=workers > 0,
        worker_init_fn=_init_worker, generator=gen,
    )

In [ ]:
%%writefile bruisekit/models.py
"""The two architectures behind one interface.

THE INTERFACE
--------------
    forward_train(x) -> (logits[B,1,H,W], aux_logits[B,1,H,W] | None)
    forward(x)       -> logits[B,1,H,W]

x is RAW [0,1] pixels (see data.py). Each wrapper applies its own pixel scale.
Both models emit ONE bruise logit at full input resolution, so every downstream
consumer -- loss, sweep, metric, benchmark -- is architecture-blind. Nothing
below this line ever branches on a model's name.

WHY YOLO IS BUILT WITH nc=1 (not nc=2)
---------------------------------------
The pretrained yolo26n-sem.pt has nc=19 (Cityscapes). Rebuilding the head with
nc=2 gives [B,2,H,W], from which the bruise logit is z1-z0 (2-class softmax and
sigmoid-of-the-difference are the same function). Rebuilding with nc=1 gives that
single logit directly, and Ultralytics' own loss supports it (nn.BCEWithLogitsLoss
for nc==1). One less transformation, one less place to get the sign wrong, and
structurally identical to SegFormer's 1-channel head. `.load()` transfers 360/364
tensors -- only the head is new, exactly mirroring SegFormer's randomly-initialised
1-class head on a pretrained backbone.

WHY THE HEAD GETS 10x THE BACKBONE'S LR (both architectures)
-------------------------------------------------------------
The backbone is pretrained and already has good features; a conservative LR
preserves them. The head is randomly initialised and has to catch up. This is the
SegFormer paper's recipe (Xie et al. 2021) and it is applied to YOLO here too --
not because YOLO's paper says so, but because holding the recipe fixed across
architectures is the entire point of an apple-to-apple comparison.
"""
from __future__ import annotations

import torch
import torch.nn as nn
import torch.nn.functional as F

IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD = (0.229, 0.224, 0.225)


class SegFormerNet(nn.Module):
    """HuggingFace SegFormer with a 1-class head. Input scale: ImageNet."""

    def __init__(self, pretrained_dir: str):
        super().__init__()
        from transformers import SegformerForSemanticSegmentation
        self.net = SegformerForSemanticSegmentation.from_pretrained(
            pretrained_dir, num_labels=1, ignore_mismatched_sizes=True)
        # Buffers (not constants) so they move with .to(device) and are saved with
        # the module -- the pixel scale travels with the weights it belongs to.
        self.register_buffer("mean", torch.tensor(IMAGENET_MEAN).view(1, 3, 1, 1))
        self.register_buffer("std", torch.tensor(IMAGENET_STD).view(1, 3, 1, 1))

    @property
    def backbone(self):
        return self.net.segformer

    @property
    def head(self):
        return self.net.decode_head

    def forward_train(self, x):
        x = (x - self.mean) / self.std                      # [0,1] -> ImageNet
        logits = self.net(pixel_values=x).logits            # [B,1,H/4,W/4]
        logits = F.interpolate(logits, size=x.shape[-2:], mode="bilinear", align_corners=False)
        return logits, None                                 # SegFormer has no aux head

    def forward(self, x):
        return self.forward_train(x)[0]


class YoloSemNet(nn.Module):
    """Ultralytics YOLO26n-sem with an nc=1 head. Input scale: /255 (i.e. x as-is).

    In train() the underlying module returns (main[B,1,H/8,W/8], aux[B,1,H/16,W/16]);
    in eval() it returns only the main tensor. Both are upsampled to input
    resolution here so callers never see the stride.

    NOTE the stride: YOLO's main head is at H/8 (80x80 for a 640 input) against
    SegFormer's H/4 (160x160). YOLO predicts at a quarter of SegFormer's spatial
    resolution in area, which is an architectural ceiling on boundary precision,
    not something training can fix. Worth stating in the paper.
    """

    def __init__(self, pretrained_pt: str):
        super().__init__()
        from ultralytics import YOLO
        from ultralytics.nn.tasks import SemanticSegmentationModel

        pretrained = YOLO(str(pretrained_pt))
        self.net = SemanticSegmentationModel("yolo26n-sem.yaml", nc=1, ch=3, verbose=False)
        self.net.load(pretrained.model)     # transfers the backbone; head stays random
        del pretrained

    @property
    def backbone(self):
        return self.net.model[:-1]

    @property
    def head(self):
        return self.net.model[-1]

    def forward_train(self, x):
        out = self.net(x)                                   # x already /255
        if isinstance(out, (list, tuple)):
            main, aux = out[0], out[1]
        else:
            main, aux = out, None
        size = x.shape[-2:]
        main = F.interpolate(main.float(), size=size, mode="bilinear", align_corners=False)
        if aux is not None:
            aux = F.interpolate(aux.float(), size=size, mode="bilinear", align_corners=False)
        return main, aux

    def forward(self, x):
        return self.forward_train(x)[0]


def build_model(arch: str, size: str, paths: dict) -> nn.Module:
    if arch == "segformer":
        return SegFormerNet(paths[f"segformer_{size}"])
    if arch == "yolo":
        return YoloSemNet(paths["yolo"])
    raise ValueError(f"unknown arch: {arch}")


def count_params(model: nn.Module) -> int:
    return sum(p.numel() for p in model.parameters())


def build_param_groups(model, backbone_lr: float, head_lr: float, weight_decay: float):
    """Backbone/head LR split + no weight decay on norms and biases.

    No decay on 1-D params (norm weights, biases): decaying a normalisation scale
    or a bias shrinks it toward zero with no regularising benefit -- standard in
    every transformer recipe including SegFormer's.

    Membership is by id(), not by name: the two architectures name their
    parameters completely differently, and a name-prefix rule would silently put
    every YOLO parameter in the wrong group.
    """
    backbone_ids = {id(p) for p in model.backbone.parameters()}
    groups = {
        "backbone_decay":    {"params": [], "lr": backbone_lr, "weight_decay": weight_decay},
        "backbone_no_decay": {"params": [], "lr": backbone_lr, "weight_decay": 0.0},
        "head_decay":        {"params": [], "lr": head_lr,     "weight_decay": weight_decay},
        "head_no_decay":     {"params": [], "lr": head_lr,     "weight_decay": 0.0},
    }
    for name, p in model.named_parameters():
        if not p.requires_grad or name in ("mean", "std"):
            continue
        where = "backbone" if id(p) in backbone_ids else "head"
        decay = "_no_decay" if (p.ndim <= 1 or "norm" in name.lower() or "bias" in name.lower()) else "_decay"
        groups[where + decay]["params"].append(p)

    out = [g for g in groups.values() if g["params"]]
    n_grouped = sum(len(g["params"]) for g in out)
    n_total = sum(1 for n, p in model.named_parameters() if p.requires_grad and n not in ("mean", "std"))
    assert n_grouped == n_total, f"param grouping lost {n_total - n_grouped} tensors"
    return out

In [ ]:
%%writefile bruisekit/losses.py
"""Losses. One supervised loss for every model; one distillation fusion.

WHY Dice+BCE
-------------
BCE alone is dominated by the background: bruises cover ~4.7% of pixels, so a
model predicting all-background already scores well on BCE and gets almost no
gradient toward the bruise. The Dice term is scale-invariant with respect to
object size and supplies gradient proportional to overlap, which is what we
actually report. Summing them is the standard combination for imbalanced medical
segmentation (Milletari et al. 2016 V-Net for Dice; Drozdzal et al. 2016 for the
combination), and it is also what Ultralytics' own SemanticSegmentationLoss does
(BCEWithLogits + binary Dice), so using it for both architectures does not
disadvantage YOLO relative to its native recipe.

WHAT THE DISTILLATION LOSS IS, AND WHAT IT IS NOT
--------------------------------------------------
    loss = alpha * DiceBCE(student_logits, GT)
         + (1 - alpha) * BCE(student_logits, sigmoid(teacher_logits / T_cal))

This is CALIBRATED SOFT-TARGET DISTILLATION, not Hinton et al. (2015) KD. The
difference matters for the paper and should not be papered over:

  - Hinton's KD divides BOTH the student's and the teacher's logits by a shared
    temperature T and multiplies the soft term by T^2 (to keep the soft gradient's
    magnitude comparable to the hard term's as T varies).
  - Here the student's logits are NOT temperature-scaled, and there is no T^2.
    T_cal is not a KD knob at all: it is the temperature fitted by NLL on val
    (Guo et al. 2017) that makes the teacher's probabilities CALIBRATED.

So the teacher is used as a better-calibrated estimate of P(bruise | pixel), and
the student regresses onto that probability. The justification is Menon et al.
(2021), "A statistical perspective on distillation": distillation helps to the
extent the teacher approximates the Bayes class-probability, which is exactly
what calibration improves. Do NOT cite Hinton for this formula as written.
"""
from __future__ import annotations

import torch
import torch.nn as nn
import torch.nn.functional as F


class DiceBCELoss(nn.Module):
    """BCE + (1 - mean per-image soft Dice).

    Dice is computed PER IMAGE and then averaged, not pooled over the batch. A
    batch-pooled Dice lets one large bruise dominate the whole batch's gradient
    and lets an image with no predicted pixels hide inside a batch that scored
    well -- and per-image is what the reported metric does, so the loss and the
    metric agree.
    """

    def __init__(self, smooth: float = 1.0):
        super().__init__()
        self.smooth = smooth

    def forward(self, logits, target):
        bce = F.binary_cross_entropy_with_logits(logits, target)
        prob = torch.sigmoid(logits)
        inter = (prob * target).sum(dim=(1, 2, 3))
        denom = prob.sum(dim=(1, 2, 3)) + target.sum(dim=(1, 2, 3))
        dice = (2 * inter + self.smooth) / (denom + self.smooth)
        return bce + (1.0 - dice.mean())


class SupervisedLoss(nn.Module):
    """DiceBCE on the main head + aux_weight * BCE on the auxiliary head.

    The aux term applies only to YOLO (SegFormer returns aux=None). 0.4 is
    Ultralytics' own weight for its semantic aux head; keeping their value means
    the aux head is supervised as its designers intended even though the rest of
    the recipe is ours.
    """

    def __init__(self, aux_weight: float = 0.4):
        super().__init__()
        self.main = DiceBCELoss()
        self.aux_weight = aux_weight

    def forward(self, logits, aux_logits, target):
        loss = self.main(logits, target)
        if aux_logits is not None:
            loss = loss + self.aux_weight * F.binary_cross_entropy_with_logits(aux_logits, target)
        return loss


class DistillLoss(nn.Module):
    """alpha * supervised(GT) + (1-alpha) * BCE(student, calibrated teacher prob).

    See the module docstring for what this is and is not.
    """

    def __init__(self, alpha: float = 0.5, aux_weight: float = 0.4):
        super().__init__()
        self.alpha = alpha
        self.sup = SupervisedLoss(aux_weight)

    def forward(self, logits, aux_logits, target, teacher_prob):
        hard = self.sup(logits, aux_logits, target)
        soft = F.binary_cross_entropy_with_logits(logits, teacher_prob)
        return self.alpha * hard + (1.0 - self.alpha) * soft

In [ ]:
%%writefile bruisekit/metrics.py
"""Metrics: Dice/IoU per image, and threshold-free AP for model selection."""
from __future__ import annotations

import numpy as np
import pandas as pd
import torch


def dice_np(pred: np.ndarray, gt: np.ndarray) -> float:
    pred, gt = pred.astype(bool), gt.astype(bool)
    denom = pred.sum() + gt.sum()
    return 1.0 if denom == 0 else float(2 * np.logical_and(pred, gt).sum() / denom)


def iou_np(pred: np.ndarray, gt: np.ndarray) -> float:
    pred, gt = pred.astype(bool), gt.astype(bool)
    union = np.logical_or(pred, gt).sum()
    return 1.0 if union == 0 else float(np.logical_and(pred, gt).sum() / union)


def compute_image_row(pred: np.ndarray, gt: np.ndarray, stem: str) -> dict:
    pred_b, gt_b = pred.astype(bool), gt.astype(bool)
    tp = int(np.logical_and(pred_b, gt_b).sum())
    fp = int(np.logical_and(pred_b, ~gt_b).sum())
    fn = int(np.logical_and(~pred_b, gt_b).sum())
    return {
        "stem": stem,
        "dice": dice_np(pred, gt), "iou": iou_np(pred, gt),
        "precision": 1.0 if tp + fp == 0 else tp / (tp + fp),
        "recall": 1.0 if tp + fn == 0 else tp / (tp + fn),
        "pred_positive_pixels": int(pred_b.sum()),
        "gt_positive_pixels": int(gt_b.sum()),
    }


def summarize(rows: list[dict]) -> dict:
    df = pd.DataFrame(rows)
    # "Complete miss" = the model output ZERO pixels on an image that has a bruise.
    # This is the metric that separates the models by more than label noise, and for
    # an injury-documentation tool it is the one that actually matters: a blank mask
    # is a missed injury. Reported as a first-class number, not buried in the tail.
    miss = (df["pred_positive_pixels"] == 0) & (df["gt_positive_pixels"] > 0)
    return {
        "n_images": int(len(df)),
        "mean_dice": float(df["dice"].mean()),
        "median_dice": float(df["dice"].median()),
        "mean_iou": float(df["iou"].mean()),
        "median_iou": float(df["iou"].median()),
        "mean_precision": float(df["precision"].mean()),
        "mean_recall": float(df["recall"].mean()),
        "complete_miss_count": int(miss.sum()),
        "complete_miss_rate": float(miss.mean()),
    }


class BinnedAP:
    """Threshold-free average precision over pixels, via probability histograms.

    WHY AP IS THE MODEL-SELECTION METRIC
    -------------------------------------
    The old pipeline saved best_model.pt by val Dice AT A FIXED 0.5 -- but the
    threshold is re-fitted afterwards anyway, and the fitted operating points are
    nowhere near 0.5 (YOLO's lands around 0.18). So 0.5-Dice selection asks "which
    epoch is best at an operating point we will not use?" and can pick the wrong
    epoch for any model whose calibration drifts during training. AP integrates
    over ALL thresholds, so the epoch choice cannot be biased by one arbitrary cut.

    WHY HISTOGRAMS AND NOT sklearn.average_precision_score
    -------------------------------------------------------
    134 val images x 640 x 640 = 55M pixels. Sorting 55M floats per epoch costs
    seconds of wall-clock and ~450 MB. Binning into 4096 buckets on the GPU makes
    the whole thing O(bins) in memory and effectively free, at a quantisation
    error of ~1/4096 in probability -- three orders of magnitude below the
    epoch-to-epoch differences it has to rank.
    """

    def __init__(self, bins: int = 4096, device: str = "cuda"):
        self.bins = bins
        self.pos = torch.zeros(bins, dtype=torch.float64, device=device)
        self.neg = torch.zeros(bins, dtype=torch.float64, device=device)

    @torch.no_grad()
    def update(self, prob: torch.Tensor, gt: torch.Tensor) -> None:
        p = prob.reshape(-1).float().clamp(0, 1)
        g = gt.reshape(-1) > 0.5
        idx = (p * (self.bins - 1)).round().long()
        self.pos += torch.bincount(idx[g], minlength=self.bins).double()
        self.neg += torch.bincount(idx[~g], minlength=self.bins).double()

    def compute(self) -> float:
        """AP = sum over thresholds of (recall_k - recall_{k-1}) * precision_k."""
        total_pos = self.pos.sum()
        if total_pos == 0:
            return float("nan")
        # Walk bins from the highest probability downward: each step admits one more
        # bucket as "predicted positive", which is exactly sweeping the threshold down.
        tp = torch.cumsum(self.pos.flip(0), dim=0)
        fp = torch.cumsum(self.neg.flip(0), dim=0)
        precision = tp / (tp + fp).clamp_min(1e-12)
        recall = tp / total_pos
        d_recall = torch.diff(recall, prepend=torch.zeros(1, dtype=recall.dtype, device=recall.device))
        return float((d_recall * precision).sum())

In [ ]:
%%writefile bruisekit/engine.py
"""The training loop. One loop for every model, and it survives session death."""
from __future__ import annotations

import json
import os
import random
import shutil
import time
from copy import deepcopy
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from tqdm.auto import tqdm

from bruisekit.data import make_loader
from bruisekit.losses import DistillLoss, SupervisedLoss
from bruisekit.metrics import BinnedAP
from bruisekit.models import build_model, build_param_groups, count_params


def seed_everything(seed: int) -> None:
    """Seed every RNG that touches training.

    cudnn.deterministic without use_deterministic_algorithms(True): the latter
    makes several ops here (bilinear interpolate backward, scatter_add) raise
    instead of run. Bitwise GPU determinism is therefore NOT claimed -- which is
    precisely why this notebook runs three seeds and reports spread rather than
    pretending one run is the answer.
    """
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


def lr_multiplier(step: int, total_steps: int, warmup_steps: int, power: float = 1.0) -> float:
    """Linear warmup 0->1, then poly decay 1->0. SegFormer's schedule (Xie 2021)."""
    if step <= warmup_steps:
        return step / max(1, warmup_steps)
    progress = min(1.0, (step - warmup_steps) / max(1, total_steps - warmup_steps))
    return (1.0 - progress) ** power


def resolve_micro_batch(model, cfg, device, teacher=None) -> tuple[int, int]:
    """Choose (micro_batch, accum_steps) by probing what actually fits in VRAM.

    Two policies, selected by cfg["batch_mode"]:

    "matched" (the paper recipe) -- the probe is CAPPED AT effective_batch, and
    accumulation makes up any shortfall, so EVERY model trains at the same effective
    batch (8) and the same number of optimizer steps regardless of GPU. This is the
    fix for the old pipeline's central bug: the probe used to return 32 for the small
    models while accum collapsed to 1, so B2 trained at batch 8 / 8700 steps and both
    B0s at batch 32 / 2100 steps at the same LR -- making every "identical
    hyperparameters" claim between B0 and B2 false. A T4 just uses more accumulation
    than an A100 and lands in the same place.

    "per_model" -- the probe is NOT capped; each model uses the largest power-of-2
    batch its own size allows (accum = 1). B2 stays ~8, B0/YOLO climb much higher.
    Faster and better GPU utilisation, but the models NO LONGER share an effective
    batch or a step count, so this policy is not recipe-matched across model sizes
    (see the per-model notebook's §1/§6 warning). Same LR is kept deliberately (no
    linear-scaling adjustment), so the small models take fewer, larger steps.

    The probe runs on a DEEPCOPY: it does real forward/backward/step, and probing the
    live model would perturb the pretrained weights before training starts by a
    model-dependent amount.
    """
    mode = cfg.get("batch_mode", "matched")
    effective = cfg["effective_batch"]
    if not torch.cuda.is_available():
        # No GPU to probe: matched keeps its fixed effective batch via accumulation;
        # per_model has no target to hit, so fall back to a single sample.
        return (1, effective) if mode == "matched" else (1, 1)

    # matched caps the probe at effective_batch (accum fills the rest); per_model
    # lets it climb to max_probe_batch, whatever the model can hold.
    probe_ceiling = min(cfg["max_probe_batch"], effective) if mode == "matched" else cfg["max_probe_batch"]

    probe_model = deepcopy(model).to(device)
    probe_opt = torch.optim.SGD(probe_model.parameters(), lr=1e-9)
    scaler = torch.amp.GradScaler("cuda") if cfg["amp"] else None
    total_vram = torch.cuda.get_device_properties(device).total_memory

    chosen, batch = 1, 1
    while batch <= probe_ceiling:
        try:
            torch.cuda.empty_cache()
            torch.cuda.reset_peak_memory_stats(device)
            x = torch.rand(batch, 3, cfg["img_size"], cfg["img_size"], device=device)
            y = torch.randint(0, 2, (batch, 1, cfg["img_size"], cfg["img_size"]), device=device).float()
            probe_model.train()
            probe_opt.zero_grad(set_to_none=True)
            with torch.amp.autocast("cuda", enabled=cfg["amp"]):
                # The teacher runs on every real training step too; excluding it here
                # would choose a batch that OOMs the moment distillation starts.
                if teacher is not None:
                    _ = teacher(x)
                logits, aux = probe_model.forward_train(x)
                loss = nn.functional.binary_cross_entropy_with_logits(logits, y)
                if aux is not None:
                    loss = loss + nn.functional.binary_cross_entropy_with_logits(aux, y)
            if scaler is not None:
                scaler.scale(loss).backward(); scaler.step(probe_opt); scaler.update()
            else:
                loss.backward(); probe_opt.step()
            frac = torch.cuda.max_memory_reserved(device) / total_vram
            del x, y, logits, loss
            # 0.75 leaves headroom for the real loss (Dice+BCE+aux), augmentation
            # variance and the checkpoint write -- the probe only sees a plain BCE.
            if frac > cfg.get("vram_target", 0.75):
                break
            chosen = batch
            batch *= 2
        except torch.cuda.OutOfMemoryError:
            break

    del probe_model, probe_opt
    torch.cuda.empty_cache()

    micro = max(1, chosen)
    if mode == "per_model":
        return micro, 1                    # use the probed batch directly, no accumulation
    while effective % micro != 0:          # matched: keep micro*accum EXACT, never approximate
        micro -= 1
    accum = effective // micro
    assert micro * accum == effective, f"{micro} x {accum} != {effective}"
    return micro, accum


def load_teacher(teacher_dir: Path, paths: dict, device, amp: bool):
    """Load a trained B2 as a frozen, calibrated soft-label generator.

    Returns a callable so the training loop never learns anything about the
    teacher's architecture -- it just calls teacher(x) and gets a probability map.
    """
    from bruisekit.models import build_model as _bm
    ckpt = teacher_dir / "best.pt"
    if not ckpt.exists():
        raise FileNotFoundError(f"Teacher not trained yet: {ckpt}")

    model = _bm("segformer", "b2", paths).to(device)
    model.load_state_dict(torch.load(str(ckpt), map_location=device, weights_only=True))
    model.eval()
    for p in model.parameters():
        p.requires_grad_(False)

    temperature = json.loads((teacher_dir / "calibration.json").read_text())["temperature"]

    def teacher_fn(x):
        # no_grad: the teacher is frozen. This is the single largest memory saving
        # in the distillation setup -- without it we would build a backward graph
        # through 27M parameters we never update.
        with torch.no_grad(), torch.amp.autocast("cuda", enabled=amp):
            logits = model(x)
        return torch.sigmoid(logits.float() / temperature)

    teacher_fn.temperature = temperature
    return teacher_fn


def calibrate_temperature(model, loader, device, amp: bool) -> dict:
    """Fit the scalar T minimising val NLL of sigmoid(z/T). Guo et al. (2017).

    WHY: BCE drives a trained model's logits toward +-inf, because sigmoid(+-inf)
    is a perfect loss. The teacher's probability histogram ends up nearly binary,
    so sigmoid(z_teacher) as a soft label is almost indistinguishable from the hard
    GT -- which defeats the point of soft-label distillation. Dividing by T > 1
    pulls saturated logits back into sigmoid's responsive region, so the student
    can see where the teacher is UNCERTAIN, which is the information distillation
    is supposed to transfer.

    Optimises log(T), not T: keeps T > 0 automatically without a constraint, and
    avoids the singularity at T = 0.
    L-BFGS, not SGD: one scalar over a fixed dataset -- second-order curvature
    converges in ~10 iterations where SGD needs hundreds.
    """
    model.eval()
    logits_all, targets_all = [], []
    with torch.no_grad():
        for x, y, _ in tqdm(loader, desc="collect val logits", leave=False):
            x = x.to(device, non_blocking=True)
            with torch.amp.autocast("cuda", enabled=amp):
                z = model(x)
            # Downsample 4x before storing: calibration fits ONE scalar, and a
            # regular 1-in-16 pixel subsample of 55M pixels estimates it to far
            # more precision than it is worth -- while keeping this in RAM.
            logits_all.append(z.float()[..., ::4, ::4].cpu())
            targets_all.append(y[..., ::4, ::4].cpu())

    logits = torch.cat(logits_all)
    targets = torch.cat(targets_all)
    nll_before = float(torch.nn.functional.binary_cross_entropy_with_logits(logits, targets))

    log_t = torch.zeros(1, requires_grad=True)
    opt = torch.optim.LBFGS([log_t], lr=0.05, max_iter=100)

    def closure():
        opt.zero_grad()
        loss = torch.nn.functional.binary_cross_entropy_with_logits(logits / torch.exp(log_t), targets)
        loss.backward()
        return loss

    opt.step(closure)
    temperature = float(torch.exp(log_t).item())
    nll_after = float(torch.nn.functional.binary_cross_entropy_with_logits(logits / temperature, targets))

    # A well-trained BCE model is OVER-confident, so T lands a little above 1
    # (this project's B2 previously calibrated to 1.84). Anything far outside that
    # is a symptom, not a temperature:
    #   T >> 10  -> the logits are near zero, i.e. the model is barely trained and
    #               calibration is degenerately flattening an already-flat output.
    #               Distilling from it would teach the student a constant 0.5.
    #   T < 0.5  -> the model is UNDER-confident, which BCE training does not
    #               normally produce -- suspect the checkpoint or the val split.
    # Loud warning rather than a raise: the number is still recorded and the run can
    # proceed, but this must never pass silently into a paper.
    if not (0.5 <= temperature <= 10.0):
        print(f"  !! WARNING: calibrated T={temperature:.3f} is outside the plausible "
              f"[0.5, 10] range. The teacher is probably under-trained (near-zero "
              f"logits) -- check its val AP before trusting any distilled student.")

    return {"temperature": temperature, "nll_before": nll_before, "nll_after": nll_after,
            "n_pixels_used": int(targets.numel()), "plausible": bool(0.5 <= temperature <= 10.0)}


@torch.no_grad()
def eval_ap(model, loader, device, amp: bool) -> float:
    """Threshold-free val AP -- the model-selection metric. See metrics.BinnedAP."""
    model.eval()
    ap = BinnedAP(device=str(device))
    for x, y, _ in loader:
        x, y = x.to(device, non_blocking=True), y.to(device, non_blocking=True)
        with torch.amp.autocast("cuda", enabled=amp):
            logits = model(x)
        ap.update(torch.sigmoid(logits.float()), y)
    return ap.compute()


def _atomic_save(obj, dest: Path) -> None:
    """Write to a temp file, then rename.

    Drive rename is atomic; a plain torch.save straight to Drive that is killed
    halfway leaves a TRUNCATED checkpoint, and the next session then dies trying
    to resume from it -- turning one lost session into a lost run.
    """
    tmp = dest.with_suffix(dest.suffix + ".tmp")
    torch.save(obj, tmp)
    os.replace(tmp, dest)


def train_run(run_id: str, spec: dict, seed: int, cfg: dict, paths: dict,
              manifests: dict, root: Path, runs_dir: Path, device) -> dict:
    """Train one (model, seed). Idempotent and resumable.

    RESUME CONTRACT
    ----------------
    - DONE.json exists          -> return immediately, touch nothing.
    - resume.pt exists          -> restore model+optimizer+scaler+epoch+best and continue.
    - neither                   -> fresh start.

    resume.pt is written every cfg["drive_sync_every"] epochs (and always on the
    final epoch), so a disconnect costs at most that many epochs. It is saved even
    on epochs where val AP did NOT improve: the best weights live in best.pt, but
    resuming must continue from where training actually WAS, not from the last
    good epoch, or the LR schedule and optimizer moments silently rewind.
    """
    run_dir = runs_dir / run_id
    run_dir.mkdir(parents=True, exist_ok=True)
    done_file = run_dir / "DONE.json"
    if done_file.exists():
        return {"run_id": run_id, "status": "skipped", **json.loads(done_file.read_text())}

    seed_everything(seed)
    amp = cfg["amp"]

    model = build_model(spec["arch"], spec["size"], paths).to(device)

    teacher = None
    if spec["distill"]:
        # Same-seed teacher: the distilled run's spread over seeds then includes the
        # teacher's own variance, which is part of the pipeline being measured.
        teacher = load_teacher(runs_dir / f"segformer_b2_teacher__seed{seed}", paths, device, amp)

    micro, accum = resolve_micro_batch(model, cfg, device, teacher)

    train_loader = make_loader(manifests["train"], root, cfg["img_size"], micro,
                               training=True, workers=cfg["workers"], seed=seed)
    val_loader = make_loader(manifests["val"], root, cfg["img_size"], micro,
                             training=False, workers=cfg["workers"], seed=seed)

    param_groups = build_param_groups(model, cfg["backbone_lr"], cfg["head_lr"], cfg["weight_decay"])
    optimizer = torch.optim.AdamW(param_groups, betas=tuple(cfg["betas"]))
    peak_lrs = [g["lr"] for g in param_groups]
    scaler = torch.amp.GradScaler("cuda") if amp else None

    steps_per_epoch = max(1, len(train_loader) // accum)
    total_steps = steps_per_epoch * cfg["epochs"]
    warmup_steps = max(1, int(total_steps * cfg["warmup_fraction"]))

    criterion = (DistillLoss(cfg["alpha"], cfg["aux_weight"]) if teacher is not None
                 else SupervisedLoss(cfg["aux_weight"]))

    start_epoch, best_ap, patience, global_step, history = 1, float("-inf"), 0, 0, []
    resume_path = run_dir / "resume.pt"
    if resume_path.exists():
        st = torch.load(str(resume_path), map_location=device, weights_only=False)
        model.load_state_dict(st["model"])
        optimizer.load_state_dict(st["optimizer"])
        if scaler is not None and st.get("scaler"):
            scaler.load_state_dict(st["scaler"])
        start_epoch, best_ap = st["epoch"] + 1, st["best_ap"]
        patience, global_step, history = st["patience"], st["global_step"], st["history"]
        print(f"  [resume] {run_id} from epoch {start_epoch} (best_ap={best_ap:.4f})")
        del st

    (run_dir / "config.json").write_text(json.dumps({
        "run_id": run_id, "seed": seed, **spec,
        "micro_batch": micro, "accum_steps": accum, "effective_batch": micro * accum,
        "total_steps": total_steps, "warmup_steps": warmup_steps,
        "params": count_params(model),
        "alpha": cfg["alpha"] if spec["distill"] else None,
        "teacher_temperature": getattr(teacher, "temperature", None),
    }, indent=2))

    for epoch in range(start_epoch, cfg["epochs"] + 1):
        model.train()
        optimizer.zero_grad(set_to_none=True)
        running, t0 = 0.0, time.time()

        for step, (x, y, _) in enumerate(tqdm(train_loader, desc=f"{run_id} e{epoch}", leave=False)):
            x = x.to(device, non_blocking=True)
            y = y.to(device, non_blocking=True)

            with torch.amp.autocast("cuda", enabled=amp):
                # Teacher FIRST, inside its own no_grad: its activations are freed
                # before the student's backward graph is built. Student-first would
                # hold both alive at once and roughly double peak VRAM.
                tprob = teacher(x) if teacher is not None else None
                logits, aux = model.forward_train(x)
                loss = (criterion(logits, aux, y, tprob) if teacher is not None
                        else criterion(logits, aux, y))
                loss = loss / accum

            if scaler is not None:
                scaler.scale(loss).backward()
            else:
                loss.backward()
            running += loss.item() * accum

            if (step + 1) % accum == 0 or (step + 1) == len(train_loader):
                global_step += 1
                # LR is set per OPTIMIZER STEP, not per micro-batch: the schedule is
                # defined over gradient updates, and updating it per micro-batch would
                # advance it accum_steps times too fast.
                mult = lr_multiplier(global_step, total_steps, warmup_steps, cfg["poly_power"])
                for g, peak in zip(optimizer.param_groups, peak_lrs):
                    g["lr"] = peak * mult
                if scaler is not None:
                    scaler.unscale_(optimizer)   # real gradients before clipping
                    nn.utils.clip_grad_norm_(model.parameters(), cfg["gradient_clip"])
                    scaler.step(optimizer)
                    scaler.update()
                else:
                    nn.utils.clip_grad_norm_(model.parameters(), cfg["gradient_clip"])
                    optimizer.step()
                optimizer.zero_grad(set_to_none=True)

        val_ap = eval_ap(model, val_loader, device, amp)
        train_loss = running / max(1, len(train_loader))
        cur_lr = peak_lrs[0] * lr_multiplier(global_step, total_steps, warmup_steps, cfg["poly_power"])
        history.append({"epoch": epoch, "train_loss": train_loss, "val_ap": val_ap,
                        "backbone_lr": cur_lr, "sec": round(time.time() - t0, 1)})
        pd.DataFrame(history).to_csv(run_dir / "history.csv", index=False)

        if val_ap > best_ap:
            best_ap, patience = val_ap, 0
            _atomic_save(model.state_dict(), run_dir / "best.pt")
            flag = " *"
        else:
            patience += 1
            flag = ""
        print(f"  {run_id} e{epoch:3d} loss={train_loss:.4f} val_ap={val_ap:.4f}"
              f" lr={cur_lr:.2e} {time.time()-t0:.0f}s{flag}")

        last = (epoch == cfg["epochs"]) or (patience >= cfg["patience"])
        if epoch % cfg["drive_sync_every"] == 0 or last:
            _atomic_save({"epoch": epoch, "model": model.state_dict(),
                          "optimizer": optimizer.state_dict(),
                          "scaler": scaler.state_dict() if scaler else None,
                          "best_ap": best_ap, "patience": patience,
                          "global_step": global_step, "history": history}, resume_path)
        if patience >= cfg["patience"]:
            print(f"  early stop at epoch {epoch} (patience={cfg['patience']})")
            break

    # The teacher must be calibrated before any student can distil from it, and
    # calibration needs the BEST weights -- so it happens here, not in a separate step
    # that could be forgotten or run against the wrong checkpoint.
    model.load_state_dict(torch.load(str(run_dir / "best.pt"), map_location=device, weights_only=True))
    if run_id.startswith("segformer_b2_teacher"):
        cal = calibrate_temperature(model, val_loader, device, amp)
        (run_dir / "calibration.json").write_text(json.dumps(cal, indent=2))
        print(f"  calibrated T={cal['temperature']:.4f} "
              f"(val NLL {cal['nll_before']:.4f} -> {cal['nll_after']:.4f})")

    summary = {"run_id": run_id, "seed": seed, "best_val_ap": best_ap,
               "epochs_trained": len(history), "params": count_params(model),
               "micro_batch": micro, "accum_steps": accum}
    done_file.write_text(json.dumps(summary, indent=2))
    if resume_path.exists():
        resume_path.unlink()   # training finished: the resume state is now dead weight

    del model, teacher
    torch.cuda.empty_cache()
    return {"status": "trained", **summary}

In [ ]:
%%writefile bruisekit/sweep.py
"""Fit each model's operating point on val. Never on test."""
from __future__ import annotations

import numpy as np
import pandas as pd
import torch


@torch.no_grad()
def cache_logits(model, loader, device, amp: bool):
    """One forward pass per image; keep the logits.

    The sweep tries 481 cuts. Re-running the model for each would be 481 x 134 =
    64,454 forward passes; caching the logits once makes all 481 cuts pure tensor
    arithmetic on data already on the GPU. fp16 keeps 134x640x640 at ~110 MB --
    the quantisation is ~3 decimal places on a logit, far below the resolution any
    of this can distinguish.
    """
    model.eval()
    logits, gts, stems = [], [], []
    for x, y, s in loader:
        x = x.to(device, non_blocking=True)
        with torch.amp.autocast("cuda", enabled=amp):
            z = model(x)
        logits.append(z.float().half()[:, 0])
        # GT as BOOL, never float: the sweep counts pixels, and counting is what
        # bool+int64 do exactly. See sweep_cuts for why that matters.
        gts.append((y[:, 0] > 0.5).to(device))
        stems.extend(s)
    return torch.cat(logits), torch.cat(gts), stems


def sweep_cuts(logits, gts, cuts):
    """Per-cut mean Dice, its standard error, and complete-miss rate. Vectorised.

    THE REDUCTIONS ARE EXACT, ON PURPOSE. Dice is a ratio of PIXEL COUNTS, so the
    intersection and the two cardinalities are computed as boolean masks summed to
    int64 -- exactly what metrics.dice_np does in numpy, and exact by construction.

    Doing the same reductions in fp16 (tempting, since the cached logits are fp16)
    is wrong: fp16 carries an 11-bit mantissa, and a 640x640 image sums ~410k terms,
    so the running total stops being able to represent +1 long before the sum
    finishes. Measured against the numpy implementation that drifted by ~1.5e-4 per
    cut -- small, but the tie band is selected by comparing cuts whose Dice differ
    by ~1e-3, so the error is a tenth of the signal it is used to rank. The logits
    stay fp16 (storage only; the >= comparison is exact regardless).
    """
    rows = []
    gts = gts.bool()
    gt_sum = gts.sum(dim=(1, 2))              # int64, exact
    gt_has = gt_sum > 0
    n = len(gt_sum)
    for c in cuts:
        pred = logits >= c                    # bool; comparison is exact in fp16
        inter = (pred & gts).sum(dim=(1, 2))
        pred_sum = pred.sum(dim=(1, 2))
        denom = pred_sum + gt_sum
        # denom == 0 means both prediction and GT are empty: a correct agreement,
        # scored 1.0 -- matching metrics.dice_np so the sweep and the report agree.
        dice = torch.where(denom > 0,
                           2.0 * inter.double() / denom.double().clamp_min(1.0),
                           torch.ones_like(denom, dtype=torch.float64))
        miss = ((pred_sum == 0) & gt_has).double()
        d = dice
        rows.append({
            "cut": float(c), "threshold": float(torch.sigmoid(torch.tensor(c))),
            "mean_dice": float(d.mean()),
            "se_dice": float(d.std(unbiased=True) / np.sqrt(n)),
            "complete_miss_rate": float(miss.mean()),
        })
    return pd.DataFrame(rows)


def select_cut(df: pd.DataFrame) -> dict:
    """Pick the operating point: the tie band's LOWEST-MISS cut.

    WHY A TIE BAND AT ALL
    ----------------------
    These sweeps are extraordinarily flat -- on the previous models, B2's val Dice
    moved by 0.009 across thresholds from 0.154 to 0.959. That is not a peak, it is
    noise on a plateau, and taking argmax of it fits the val set's sampling error.
    Every cut within ONE STANDARD ERROR of the peak is statistically tied, so the
    band -- not the argmax -- is the honest answer to "which cut is best?".

    WHY MISS RATE BREAKS THE TIE
    -----------------------------
    Cuts in the band are Dice-equivalent but they are NOT miss-equivalent: a lower
    cut predicts more pixels, so fewer images come back completely blank, at
    statistically identical Dice. The old rule took the band's MEDIAN cut, which
    optimises for stability -- but a blank mask is a missed injury, and the miss
    rate is the one metric that separates these models by more than label noise.
    So: minimise miss rate within the band; break remaining ties on Dice; break
    what is left on the median cut, for reproducibility.
    """
    peak = df.loc[df["mean_dice"].idxmax()]
    band = df[df["mean_dice"] >= peak["mean_dice"] - peak["se_dice"]]
    best_miss = band["complete_miss_rate"].min()
    tied = band[band["complete_miss_rate"] <= best_miss + 1e-12]
    top_dice = tied["mean_dice"].max()
    finalists = tied[tied["mean_dice"] >= top_dice - 1e-12]
    chosen = finalists.iloc[len(finalists) // 2]
    return {
        "cut": float(chosen["cut"]), "threshold": float(chosen["threshold"]),
        "val_dice_at_cut": float(chosen["mean_dice"]),
        "val_miss_at_cut": float(chosen["complete_miss_rate"]),
        "val_peak_dice": float(peak["mean_dice"]),
        "peak_cut": float(peak["cut"]),
        "band_lo_threshold": float(band["threshold"].min()),
        "band_hi_threshold": float(band["threshold"].max()),
        "band_width_cuts": int(len(band)),
        "n_cuts": int(len(df)),
    }

In [ ]:
%%writefile bruisekit/evaluate.py
"""Test scoring, fairness across skin tone, and the speed benchmark."""
from __future__ import annotations

import numpy as np
import pandas as pd
import torch
from scipy import stats

from bruisekit.metrics import compute_image_row, summarize


@torch.no_grad()
def evaluate_at_cut(model, loader, device, cut: float, amp: bool) -> tuple[pd.DataFrame, dict]:
    """Score on test at an operating point ALREADY FIXED on val. One pass, no tuning."""
    model.eval()
    rows = []
    for x, y, stems in loader:
        x = x.to(device, non_blocking=True)
        with torch.amp.autocast("cuda", enabled=amp):
            z = model(x)
        pred = (z.float()[:, 0] >= cut).cpu().numpy().astype(np.uint8)
        gt = (y[:, 0] > 0.5).numpy().astype(np.uint8)
        for i, stem in enumerate(stems):
            rows.append(compute_image_row(pred[i], gt[i], stem))
    return pd.DataFrame(rows), summarize(rows)


def bootstrap_ci(values: np.ndarray, n: int = 2000, seed: int = 0) -> tuple[float, float]:
    """Percentile bootstrap CI of the MEDIAN.

    The median, not the mean: per-image Dice is strongly bimodal (a model either
    localises the bruise or misses it completely), so the mean mixes two different
    populations. And a bootstrap, not a t-interval, because that bimodality makes
    the normal approximation wrong.
    """
    if len(values) < 2:
        return float("nan"), float("nan")
    rng = np.random.default_rng(seed)
    meds = [np.median(rng.choice(values, size=len(values), replace=True)) for _ in range(n)]
    return float(np.percentile(meds, 2.5)), float(np.percentile(meds, 97.5))


def fairness_analysis(per_image: pd.DataFrame, manifest: pd.DataFrame, model_name: str) -> dict:
    """Is performance equitable across Fitzpatrick/ITA skin-tone groups?

    THE STAKES: this is a forensic injury-documentation tool. A model that
    segments bruises well on light skin and poorly on dark skin does not have a
    metric problem, it has an evidentiary one -- it would under-document injuries
    on exactly the population most likely to need the documentation. So skin tone
    is not a robustness ablation here, it is a primary result.

    METHOD, and why each piece:
      - Groups are ITA (Individual Typology Angle, Chardon et al. 1991), the
        standard objective skin-tone measure -- computed from image pixels, not
        from a rater's Fitzpatrick guess, so it is reproducible.
      - Kruskal-Wallis across all 5 groups: an omnibus test for "does ANY group
        differ?". Non-parametric because per-image Dice is bimodal and bounded,
        so ANOVA's normality assumption fails.
      - Pairwise Mann-Whitney U, Bonferroni-corrected over the 10 pairs: with 5
        groups, uncorrected pairwise testing finds a "significant" pair ~40% of
        the time on pure noise.
      - fairness_gap = best group's median Dice - worst group's. The effect size.
        A p-value says a gap is real; only the gap says whether it matters.
    """
    df = per_image.merge(manifest[["stem", "skin_tone_category", "ita_group_index_5"]],
                         on="stem", how="left", validate="one_to_one")
    if df["ita_group_index_5"].isna().any():
        raise RuntimeError(f"{int(df['ita_group_index_5'].isna().sum())} test images have no skin-tone label")

    per_group, samples = [], []
    for gidx, g in sorted(df.groupby("ita_group_index_5"), key=lambda kv: kv[0]):
        vals = g["dice"].to_numpy()
        lo, hi = bootstrap_ci(vals)
        per_group.append({
            "model": model_name, "ita_group_index_5": int(gidx),
            "skin_tone_category": g["skin_tone_category"].iloc[0],
            "n_images": len(g), "median_dice": float(np.median(vals)),
            "iqr_dice": float(np.percentile(vals, 75) - np.percentile(vals, 25)),
            "ci95_lo": lo, "ci95_hi": hi,
            "mean_recall": float(g["recall"].mean()),
            "miss_rate": float(((g["pred_positive_pixels"] == 0) & (g["gt_positive_pixels"] > 0)).mean()),
        })
        samples.append(vals)

    H, p = stats.kruskal(*samples)
    pairwise = []
    raw = []
    pairs = [(i, j) for i in range(len(samples)) for j in range(i + 1, len(samples))]
    for i, j in pairs:
        raw.append(stats.mannwhitneyu(samples[i], samples[j], alternative="two-sided").pvalue)
    for (i, j), pv in zip(pairs, raw):
        adj = min(1.0, pv * len(pairs))    # Bonferroni over the 10 pairs
        pairwise.append({"model": model_name, "group_a": per_group[i]["skin_tone_category"],
                         "group_b": per_group[j]["skin_tone_category"],
                         "pvalue": pv, "bonferroni_p": adj, "significant": bool(adj < 0.05)})

    pg = pd.DataFrame(per_group)
    best, worst = pg.loc[pg["median_dice"].idxmax()], pg.loc[pg["median_dice"].idxmin()]
    stats_row = {
        "model": model_name, "kruskal_H": float(H), "kruskal_p": float(p),
        "significant": bool(p < 0.05),
        "fairness_gap": float(best["median_dice"] - worst["median_dice"]),
        "best_group": best["skin_tone_category"], "worst_group": worst["skin_tone_category"],
        "max_miss_rate_gap": float(pg["miss_rate"].max() - pg["miss_rate"].min()),
    }
    return {"per_group": pg, "pairwise": pd.DataFrame(pairwise), "stats": stats_row}


@torch.no_grad()
def benchmark_speed(model, images, device, cut: float, repeats: int, warmup: int) -> dict:
    """Time 640-tensor-on-GPU -> mask-on-GPU. Nothing else.

    WHAT IS DELIBERATELY NOT TIMED: disk read, JPEG decode, resize, host->GPU copy,
    and GPU->host copy. Those are identical for every model (one dataloader) and
    are dominated by I/O, so including them would compress the real architectural
    differences into measurement noise. The mask never leaves the GPU.

    cuda.synchronize() around each call is not optional: CUDA is asynchronous, so
    without it this would measure how long it takes to QUEUE the work, not do it --
    which reports every model as equally, impossibly fast.
    """
    model.eval()
    for _ in range(warmup):
        _ = torch.sigmoid(model(images[:1])) >= torch.sigmoid(torch.tensor(cut, device=device))
    torch.cuda.synchronize()

    times = []
    for _ in range(repeats):
        for i in range(len(images)):
            x = images[i:i + 1]
            torch.cuda.synchronize()
            t0 = time.perf_counter()
            z = model(x)
            _ = z >= cut                      # threshold on the logit == sigmoid >= sigmoid(cut)
            torch.cuda.synchronize()
            times.append((time.perf_counter() - t0) * 1000)

    arr = np.array(times)
    return {"median_ms": float(np.median(arr)), "mean_ms": float(arr.mean()),
            "p95_ms": float(np.percentile(arr, 95)), "fps": float(1000.0 / np.median(arr)),
            "n_timed": len(arr)}


import time  # noqa: E402  (used by benchmark_speed above)

In [ ]:
%%writefile bruisekit/postopt.py
"""Reduce complete misses WITHOUT retraining. Everything here is fitted on val and
applied to test; nothing changes a trained weight.

THE PROBLEM THIS SOLVES
------------------------
A single global threshold couples two failure modes: pushing it down to stop blank
masks (complete misses) also floods the easy images with false positives, so miss-%
and Dice move together -- two sides of one knob. Lowering the threshold slides ALONG
the miss-vs-Dice curve; it cannot move the curve. The three techniques here try to
move the curve (fewer misses at the SAME Dice), and one deliberately games the miss
metric so its cost is visible for comparison:

  ensemble   -- average the 3 seeds' probability maps. A miss needs ALL three seeds
                to blank the same image, which is rare (the per-seed misses are
                different images). Free: no retraining, just averaging maps we can
                already produce. This is the honest, recommended lever.
  TTA        -- average probs over horizontal+vertical flips. Raises the probability
                on borderline images so they clear the threshold without lowering it.
  no-blank   -- if a mask is still empty, recover the most-confident region instead of
                returning blank. This GAMES the miss metric (guarantees a non-zero
                prediction whether or not anything real was found), so it is reported
                as a separate floor, never as the main method.

HOW TO READ THE RESULT (this is the whole point of the question)
----------------------------------------------------------------
Plot miss-% against Dice. A real improvement sits BELOW-AND-LEFT of the single-model
threshold-sweep curve -- lower miss at equal-or-better Dice. A point that just slid
down the same curve (Dice fell as much as miss) is the threshold in disguise, not an
improvement. Everything below fits its threshold on VAL with the same miss-tie-break
rule as the baseline, so the comparison is like-for-like.
"""
from __future__ import annotations

import numpy as np
import pandas as pd
import torch

from bruisekit.metrics import compute_image_row, summarize
from bruisekit.sweep import select_cut


@torch.no_grad()
def probs_plain(model, loader, device, amp: bool):
    """One forward pass per image -> sigmoid probability maps [N,H,W] (fp16), GT
    (bool), stems. fp16 storage keeps the whole split in memory; the >= comparison
    the sweep does is exact regardless of storage dtype."""
    model.eval()
    P, G, S = [], [], []
    use_amp = amp and device.type == "cuda"
    for x, y, s in loader:
        x = x.to(device, non_blocking=True)
        with torch.amp.autocast("cuda", enabled=use_amp):
            z = model(x)
        P.append(torch.sigmoid(z.float())[:, 0].half().cpu())
        G.append((y[:, 0] > 0.5).cpu())
        S.extend(s)
    return torch.cat(P), torch.cat(G), S


@torch.no_grad()
def probs_tta(model, loader, device, amp: bool, flips=("none", "h", "v")):
    """TTA: average sigmoid probs over identity + horizontal + vertical flip.

    Each flip is applied to the INPUT and UNDONE on the OUTPUT before averaging, so
    the maps stay pixel-aligned -- averaging misaligned maps would blur the boundary
    rather than sharpen the confidence. TTA is used identically on val (to fit the
    threshold) and test (to score); mismatching them would fit a threshold for a
    distribution the test pass never sees.
    """
    model.eval()
    P, G, S = [], [], []
    use_amp = amp and device.type == "cuda"
    for x, y, s in loader:
        x = x.to(device, non_blocking=True)
        acc = torch.zeros(x.shape[0], x.shape[2], x.shape[3], device=x.device)
        for f in flips:
            xf = torch.flip(x, [3]) if f == "h" else torch.flip(x, [2]) if f == "v" else x
            with torch.amp.autocast("cuda", enabled=use_amp):
                z = model(xf)
            p = torch.sigmoid(z.float())[:, 0]
            p = torch.flip(p, [2]) if f == "h" else torch.flip(p, [1]) if f == "v" else p
            acc = acc + p
        P.append((acc / len(flips)).half().cpu())
        G.append((y[:, 0] > 0.5).cpu())
        S.extend(s)
    return torch.cat(P), torch.cat(G), S


def mean_over_seeds(prob_list, stem_list):
    """Average probability maps across runs, aligned by stem.

    Loaders are shuffle=False so every seed already returns images in the same order,
    but this re-indexes by stem anyway rather than trusting that -- a silent order
    mismatch would average seed A's image 5 onto seed B's image 6 and quietly corrupt
    every ensemble number. Asserts the image SETS are identical first.
    """
    ref = stem_list[0]
    ref_set = set(ref)
    for sl in stem_list:
        if set(sl) != ref_set:
            raise ValueError("ensemble seeds cover different image sets")
    acc = None
    for probs, sl in zip(prob_list, stem_list):
        pos = {s: i for i, s in enumerate(sl)}
        reordered = torch.stack([probs[pos[s]] for s in ref]).float()
        acc = reordered if acc is None else acc + reordered
    return (acc / len(prob_list)).half(), list(ref)


def sweep_prob(probs, gts, thresholds):
    """Per-threshold mean Dice / SE / complete-miss on probability maps.

    Same exact-integer pixel counting as sweep.sweep_cuts (bool masks summed to
    int64), so the numbers are directly comparable to the logit-cut sweep. Emits the
    columns select_cut expects, with `cut` == `threshold` (probability space).

    The comparison is done in float32, NOT the fp16 the maps are stored in, so the
    threshold this sweep FITS on val is applied at the exact same numeric boundary
    score_prob_at() uses on test (it also compares in float32). Comparing fp16 here
    would put the val fit and the test apply on slightly different boundaries for any
    pixel sitting right at the threshold -- the same "the sweep must match the score"
    trap the logit sweep already guards against.
    """
    probs = probs.float()
    gts = gts.bool()
    gt_sum = gts.sum(dim=(1, 2))
    gt_has = gt_sum > 0
    n = len(gt_sum)
    rows = []
    for t in thresholds:
        pred = probs >= t
        inter = (pred & gts).sum(dim=(1, 2))
        ps = pred.sum(dim=(1, 2))
        den = ps + gt_sum
        dice = torch.where(den > 0, 2.0 * inter.double() / den.double().clamp_min(1.0),
                           torch.ones_like(den, dtype=torch.float64))
        miss = ((ps == 0) & gt_has).double()
        rows.append({"cut": float(t), "threshold": float(t),
                     "mean_dice": float(dice.mean()),
                     "se_dice": float(dice.std(unbiased=True) / np.sqrt(n)),
                     "complete_miss_rate": float(miss.mean())})
    return pd.DataFrame(rows)


def score_prob_at(probs, gts, stems, thr, no_blank=False, rel=0.5):
    """Score probability maps at a fixed threshold; optional no-blank floor.

    no_blank: when the thresholded mask is empty on an image that has a bruise, fall
    back to the region at >= rel * max-probability -- the most-confident blob the
    model saw. This never returns blank, which is exactly why it must be reported
    separately: it converts a genuine miss into a (possibly wrong) small prediction.
    """
    p_np = probs.float().numpy()
    g_np = gts.numpy()
    rows = []
    for i, s in enumerate(stems):
        p = p_np[i]
        pred = (p >= thr).astype("uint8")
        if no_blank and pred.sum() == 0 and p.max() > 0:
            pred = (p >= rel * float(p.max())).astype("uint8")
        rows.append(compute_image_row(pred, g_np[i].astype("uint8"), s))
    return pd.DataFrame(rows), summarize(rows)


def fit_on_val_apply_to_test(val_probs, val_gts, test_probs, test_gts, test_stems,
                             thresholds, no_blank=False):
    """The honest protocol: sweep val -> select_cut (miss-tie-break) -> score test.

    Returns (operating_point_dict, test_per_image_df, test_summary). The threshold is
    chosen ONLY from val, then applied once to test, exactly like the baseline in the
    main notebook -- so any miss/Dice difference is the technique, not a re-tuned cut.
    """
    grid = sweep_prob(val_probs, val_gts, thresholds)
    op = select_cut(grid)
    per_img, summ = score_prob_at(test_probs, test_gts, test_stems, op["threshold"], no_blank=no_blank)
    return op, per_img, summ

In [ ]:
%%writefile bruisekit/yolo_native.py
"""Native Ultralytics YOLO training + the two prediction paths.

This module deliberately uses Ultralytics (mosaic, EMA, letterbox, its own recipe) --
that is the whole point: YOLO is trained on its home turf, not on SegFormer's
transformer recipe. It is the ONLY module that imports ultralytics at train time.

TWO PREDICTION PATHS, and why both:
  native argmax   -- YOLO.predict() letterboxes to 640, argmaxes the 2-class head,
                     returns the mask at native resolution. We bring pred and GT to
                     640 (nearest) together and score. This is YOLO's best case and
                     reproduces its ~0.83 median.
  custom /255     -- pull the raw nn.Module, feed the SAME 640 stretch tensor SegFormer
                     sees (just /255, no ImageNet norm), take sigmoid(z1 - z0), and
                     sweep the threshold on val. Same geometry as SegFormer, so it is
                     the apples-to-apples-on-eval number.
"""
from __future__ import annotations

import copy
import shutil
from pathlib import Path

import cv2
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F

from bruisekit.metrics import compute_image_row, summarize
from bruisekit.sweep import select_cut


def _ul_device() -> str:
    """Ultralytics device string: '0' on GPU (Colab), 'cpu' otherwise (local tests).

    Derived, not hardcoded, so the same module runs on the A100 in Colab and on a CPU
    box for verification without a code change -- a hardcoded '0' raises on CPU.
    """
    return "0" if torch.cuda.is_available() else "cpu"


def _read_native_mask(root: Path, rel: str) -> np.ndarray:
    m = cv2.imread(str(root / rel), cv2.IMREAD_GRAYSCALE)
    if m is None:
        raise RuntimeError(f"Cannot read mask: {rel}")
    if m.ndim == 3:
        m = m[..., 0]
    return (m > 0).astype(np.uint8)


def _to_640(arr: np.ndarray) -> np.ndarray:
    return cv2.resize(arr.astype(np.uint8), (640, 640), interpolation=cv2.INTER_NEAREST)


# ── dataset build (native-res, Ultralytics semantic format) ──────────────────
def build_yolo_dataset(df, work_root: Path, out_dir: Path, split: str,
                       alpha=None, teacher_prob_fn=None, imgsz: int = 640) -> None:
    """images/<split>/ (symlink to native) + masks/<split>/ (0/1 class PNG, native res).

    For a distilled TRAIN split, the pseudo-mask fuses the teacher's native-resolution
    probability: class = (alpha*GT + (1-alpha)*teacher_prob >= 0.5). alpha < 0.5 keeps
    this non-degenerate (alpha > 0.5 would collapse it to plain GT). Val/test always use
    clean GT.
    """
    img_dir = out_dir / "images" / split; img_dir.mkdir(parents=True, exist_ok=True)
    msk_dir = out_dir / "masks" / split;  msk_dir.mkdir(parents=True, exist_ok=True)
    for _, r in df.iterrows():
        src = (work_root / r.image_path).resolve()
        dst = img_dir / Path(r.image_path).name
        if not dst.exists():
            try:
                dst.symlink_to(src)
            except OSError:
                shutil.copy2(src, dst)
        gt = _read_native_mask(work_root, r.mask_path).astype(np.float32)
        if split == "train" and teacher_prob_fn is not None:
            prob = teacher_prob_fn(work_root / r.image_path, gt.shape)
            cls = ((alpha * gt + (1.0 - alpha) * prob) >= 0.5).astype(np.uint8)
        else:
            cls = gt.astype(np.uint8)
        cv2.imwrite(str(msk_dir / f"{r.stem}.png"), cls)


def write_data_yaml(out_dir: Path, run_dir: Path) -> Path:
    import yaml
    data = {"path": str(out_dir.resolve()), "train": "images/train", "val": "images/val",
            "masks_dir": "masks", "names": {0: "background", 1: "bruise"}}
    p = run_dir / "data.yaml"
    with open(p, "w") as f:
        yaml.safe_dump(data, f, sort_keys=False)
    return p


def train_native(weights_path: str, data_yaml: Path, run_dir: Path, cfg: dict, seed: int) -> Path:
    """Native Ultralytics training. Skips if best.pt exists; resumes from last.pt if
    interrupted (Ultralytics writes last.pt every epoch, so a disconnect costs <=1 epoch)."""
    from ultralytics import YOLO
    best = run_dir / "ultralytics_runs" / "train" / "weights" / "best.pt"
    last = run_dir / "ultralytics_runs" / "train" / "weights" / "last.pt"
    if best.exists():
        return best
    if last.exists():
        YOLO(str(last)).train(resume=True)
        return best
    YOLO(weights_path).train(
        data=str(data_yaml), task="semantic", imgsz=cfg["img_size"], epochs=cfg["epochs"],
        patience=cfg["patience"], batch=cfg["yolo_batch"], workers=cfg["workers"],
        device=_ul_device(), project=str(run_dir / "ultralytics_runs"), name="train", exist_ok=True,
        optimizer=cfg["yolo_optimizer"], lrf=cfg["yolo_lrf"], cos_lr=True,
        warmup_epochs=cfg["yolo_warmup_epochs"], weight_decay=cfg["yolo_weight_decay"],
        close_mosaic=cfg["yolo_close_mosaic"], seed=seed,
    )
    return best


# ── path (a): native argmax ──────────────────────────────────────────────────
def predict_native_argmax(best_pt: Path, df, work_root: Path, imgsz: int = 640):
    """YOLO.predict() argmax, pred+GT brought to 640 together and scored."""
    from ultralytics import YOLO
    model = YOLO(str(best_pt))
    rows = []
    for _, r in df.iterrows():
        res = model.predict(str(work_root / r.image_path), imgsz=imgsz, device=_ul_device(), verbose=False)[0]
        if getattr(res, "semantic_mask", None) is not None:
            cm = res.semantic_mask.data
            cm = cm.cpu().numpy() if hasattr(cm, "cpu") else np.asarray(cm)
            pred = (cm == 1).astype(np.uint8)
        else:
            pred = np.zeros((imgsz, imgsz), np.uint8)
        gt = _read_native_mask(work_root, r.mask_path)
        rows.append(compute_image_row(_to_640(pred), _to_640(gt), str(r.stem)))
    return pd.DataFrame(rows), summarize(rows)


# ── path (b): custom /255, raw module, val-swept threshold ───────────────────
def _raw_module(best_pt: Path, device):
    from ultralytics import YOLO
    return copy.deepcopy(YOLO(str(best_pt)).model).to(device).eval()


@torch.no_grad()
def _probs_640(model, df640, cache_root: Path, device):
    """sigmoid(z1 - z0) at 640, feeding the 640-stretch /255 tensor (SegFormer geometry)."""
    P, G, S = [], [], []
    for _, r in df640.iterrows():
        bgr = cv2.imread(str(cache_root / r.image_path), cv2.IMREAD_COLOR)
        rgb = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB).astype(np.float32) / 255.0
        x = torch.from_numpy(rgb.transpose(2, 0, 1)).unsqueeze(0).to(device)
        out = model(x)
        if isinstance(out, (list, tuple)):
            out = out[0]
        if out.shape[-2:] != (640, 640):
            out = F.interpolate(out.float(), size=(640, 640), mode="bilinear", align_corners=False)
        z = out[:, 1] - out[:, 0] if out.shape[1] >= 2 else out[:, 0]
        P.append(torch.sigmoid(z)[0].half().cpu())
        m = cv2.imread(str(cache_root / r.mask_path), cv2.IMREAD_GRAYSCALE)
        if m.ndim == 3: m = m[..., 0]
        G.append(torch.from_numpy((m > 0)))
        S.append(str(r.stem))
    return torch.stack(P), torch.stack(G), S


def predict_custom_255(best_pt: Path, val640, test640, cache_root: Path, device, thresholds):
    """Sweep the probability threshold on val, apply once to test. Same scoring as SegFormer."""
    from bruisekit.postopt import sweep_prob, score_prob_at
    model = _raw_module(best_pt, device)
    vp, vg, _ = _probs_640(model, val640, cache_root, device)
    grid = sweep_prob(vp, vg, thresholds)
    op = select_cut(grid)
    tp, tg, ts = _probs_640(model, test640, cache_root, device)
    per_img, summ = score_prob_at(tp, tg, ts, op["threshold"])
    return op, per_img, summ

## §5 · Import and self-test

In [ ]:
sys.path.insert(0, "/content")
import importlib
import bruisekit.data, bruisekit.models, bruisekit.losses, bruisekit.metrics
import bruisekit.engine, bruisekit.sweep, bruisekit.evaluate, bruisekit.postopt, bruisekit.yolo_native
for m in ("data","models","losses","metrics","engine","sweep","evaluate","postopt","yolo_native"):
    importlib.reload(sys.modules[f"bruisekit.{m}"])

from bruisekit.data import make_loader
from bruisekit.engine import train_run, load_teacher
from bruisekit.evaluate import evaluate_at_cut, fairness_analysis, benchmark_speed
from bruisekit.models import build_model, count_params
from bruisekit.sweep import cache_logits, select_cut, sweep_cuts
from bruisekit.metrics import dice_np
import bruisekit.yolo_native as yn

import json, time
import numpy as np, torch, pandas as pd

PATHS = {
    "segformer_b0": str(WORK / "pretrained_weights" / "segformer_mit_b0"),
    "segformer_b2": str(WORK / "pretrained_weights" / "segformer_mit_b2"),
    "yolo":         str(WORK / "pretrained_weights" / "yolo26n-sem.pt"),
}
DEVICE = torch.device("cuda:0")

In [ ]:
# SegFormer sanity: emits [B,1,640,640], right param counts. (YOLO trains natively, so
# no YoloSemNet here.) Loader reads the 640 cache and emits raw [0,1] pixels.
_ld = make_loader(MAN640["val"].head(4), CACHE640, CFG["img_size"], 2, False, 0)
_x, _y, _s = next(iter(_ld))
assert _x.shape == (2,3,640,640) and _y.shape == (2,1,640,640), (_x.shape, _y.shape)
assert 0.0 <= float(_x.min()) and float(_x.max()) <= 1.0
for _a,_sz,_e in [("segformer","b0",3.71),("segformer","b2",27.35)]:
    _m = build_model(_a,_sz,PATHS).to(DEVICE).eval()
    with torch.no_grad(): _z = _m(_x.to(DEVICE))
    assert _z.shape==(2,1,640,640) and abs(count_params(_m)/1e6-_e)<0.05
    print(f"{_a}-{_sz} OK {count_params(_m)/1e6:.2f}M"); del _m
torch.cuda.empty_cache()

# native argmax + custom paths are exercised for real in §10; here just confirm imports.
assert hasattr(yn, "train_native") and hasattr(yn, "predict_native_argmax") and hasattr(yn, "predict_custom_255")
print("yolo_native OK; all imports resolved.")

## §6 · Train SegFormer (custom loop, per-model batch, 3 seeds)

B2 teacher first per seed (it calibrates the teacher its distilled student needs).
Resumable: `resume.pt` every 2 epochs, `DONE.json` skips finished runs.

In [ ]:
SEG_QUEUE = []
for _seed in CFG["seeds"]:
    for _name, _spec in SEGFORMER_MODELS.items():
        SEG_QUEUE.append((f"{_name}__seed{_seed}", _name, _spec, _seed))
SEG_QUEUE.sort(key=lambda r: (r[3], not r[1].startswith("segformer_b2_teacher"), r[1]))

seg_results = []
for run_id, name, spec, seed in SEG_QUEUE:
    print(f"\n{'='*70}\n{run_id}\n{'='*70}")
    cfg_run = {**CFG, "alpha": CFG["segformer_alpha"]}
    t0 = time.time()
    res = train_run(run_id, spec, seed, cfg_run, PATHS, MAN640, CACHE640, RUNS_DIR, DEVICE)
    res["minutes"] = round((time.time()-t0)/60, 1)
    seg_results.append(res)
    print(f"-> {res['status']} best_val_ap={res.get('best_val_ap', float('nan')):.4f}")
pd.DataFrame(seg_results)

## §7 · SegFormer — fit threshold on val, score on test (185)

In [ ]:
CUTS = np.linspace(CFG["cut_min"], CFG["cut_max"], CFG["cut_steps"])
seg_test_rows = []; seg_per_image = {}
for run_id, name, spec, seed in SEG_QUEUE:
    rd = RUNS_DIR / run_id
    if not (rd / "best.pt").exists(): continue
    model = build_model(spec["arch"], spec["size"], PATHS).to(DEVICE)
    model.load_state_dict(torch.load(str(rd/"best.pt"), map_location=DEVICE, weights_only=True))
    logits, gts, _ = cache_logits(model, make_loader(MAN640["val"], CACHE640, CFG["img_size"], 8, False, CFG["workers"], seed), DEVICE, CFG["amp"])
    grid = sweep_cuts(logits, gts, CUTS); sel = select_cut(grid)
    (rd/"operating_point.json").write_text(json.dumps(sel, indent=2))
    pi, summ = evaluate_at_cut(model, make_loader(MAN640["test"], CACHE640, CFG["img_size"], 8, False, CFG["workers"], seed), DEVICE, sel["cut"], CFG["amp"])
    pi.to_csv(rd/"test_per_image.csv", index=False)
    seg_per_image[run_id] = pi
    seg_test_rows.append({"run_id": run_id, "model": name, "seed": seed, "cut": sel["cut"], **summ})
    print(f"  {run_id:<32} dice={summ['mean_dice']:.4f} miss={summ['complete_miss_rate']*100:.2f}%")
    del model, logits, gts; torch.cuda.empty_cache()
SEG_TEST = pd.DataFrame(seg_test_rows)
SEG_TEST.to_csv(OUT_DIR/"segformer_test_per_seed.csv", index=False)
SEG_TEST

## §8 · Train YOLO — **native Ultralytics** (mosaic, EMA, letterbox, its own LR)

Direct and distilled, 3 seeds. Distilled uses **offline pseudo-mask KD at α=0.4**: the
same-seed calibrated B2 teacher's native-resolution probability is fused into the hard
class mask before Ultralytics ever sees it (that's the only way its trainer can consume
a teacher — see the docs). Native auto-batch. Resumes via Ultralytics' `last.pt`.

In [ ]:
def make_teacher_prob_fn(seed):
    """Native-res teacher probability from the same-seed calibrated B2, for pseudo-masks."""
    teacher = load_teacher(RUNS_DIR / f"segformer_b2_teacher__seed{seed}", PATHS, DEVICE, CFG["amp"])
    def fn(img_path, native_hw):
        bgr = cv2.imread(str(img_path), cv2.IMREAD_COLOR)
        rgb = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)
        r = cv2.resize(rgb, (CFG["img_size"], CFG["img_size"]), interpolation=cv2.INTER_LINEAR).astype(np.float32)/255.0
        mean = np.array([0.485,0.456,0.406], np.float32); std = np.array([0.229,0.224,0.225], np.float32)
        x = torch.from_numpy(((r-mean)/std).transpose(2,0,1)).unsqueeze(0).float().to(DEVICE)
        prob = teacher(x)[0,0].float().cpu().numpy()   # teacher() already applies calibrated T + sigmoid
        return np.clip(cv2.resize(prob, (native_hw[1], native_hw[0]), interpolation=cv2.INTER_LINEAR), 0, 1)
    return fn

yolo_summ = []
for seed in CFG["seeds"]:
    for name, spec in YOLO_MODELS.items():
        run_id = f"{name}__seed{seed}"; rd = RUNS_DIR / run_id; rd.mkdir(parents=True, exist_ok=True)
        best = rd / "ultralytics_runs" / "train" / "weights" / "best.pt"
        print(f"\n{'='*70}\n{run_id}  (native Ultralytics)\n{'='*70}")
        if not best.exists():
            data_dir = rd / "yolo_data"
            if not (rd / "DATASET_DONE.json").exists():
                tfn = make_teacher_prob_fn(seed) if spec["distill"] else None
                yn.build_yolo_dataset(MAN["train"], WORK, data_dir, "train",
                                      alpha=CFG["yolo_alpha"], teacher_prob_fn=tfn, imgsz=CFG["img_size"])
                yn.build_yolo_dataset(MAN["val"], WORK, data_dir, "val", imgsz=CFG["img_size"])
                (rd/"DATASET_DONE.json").write_text(json.dumps({"alpha": CFG["yolo_alpha"] if spec["distill"] else None}))
                torch.cuda.empty_cache()
            yn.write_data_yaml(data_dir, rd)
        t0 = time.time()
        best = yn.train_native(PATHS["yolo"], rd/"data.yaml", rd, CFG, seed)
        yolo_summ.append({"run_id": run_id, "trained": best.exists(), "minutes": round((time.time()-t0)/60,1)})
        print(f"-> best.pt exists: {best.exists()}")
pd.DataFrame(yolo_summ)

## §9 · Evaluate YOLO **two ways** on all 185 test images

- **native argmax** — `.predict()`, YOLO's home turf (reproduces ~0.83 median)
- **custom /255** — raw module, same 640 geometry as SegFormer, threshold swept on val

In [ ]:
PROB_THR = np.linspace(CFG["prob_min"], CFG["prob_max"], CFG["prob_steps"])
yolo_test_rows = []; yolo_per_image = {}
for seed in CFG["seeds"]:
    for name, spec in YOLO_MODELS.items():
        run_id = f"{name}__seed{seed}"; rd = RUNS_DIR / run_id
        best = rd / "ultralytics_runs" / "train" / "weights" / "best.pt"
        if not best.exists(): continue

        # (a) native argmax
        pi_a, summ_a = yn.predict_native_argmax(best, MAN["test"], WORK, CFG["img_size"])
        pi_a.to_csv(rd/"test_per_image_native_argmax.csv", index=False)
        yolo_per_image[f"{run_id}::native_argmax"] = pi_a
        yolo_test_rows.append({"run_id": run_id, "model": name, "seed": seed, "path": "native_argmax", **summ_a})

        # (b) custom /255 (swept on val)
        op, pi_b, summ_b = yn.predict_custom_255(best, MAN640["val"], MAN640["test"], CACHE640, DEVICE, PROB_THR)
        pi_b.to_csv(rd/"test_per_image_custom255.csv", index=False)
        yolo_per_image[f"{run_id}::custom255"] = pi_b
        yolo_test_rows.append({"run_id": run_id, "model": name, "seed": seed, "path": "custom255", "threshold": op["threshold"], **summ_b})
        print(f"  {run_id:<28} argmax dice={summ_a['mean_dice']:.4f} miss={summ_a['complete_miss_rate']*100:.1f}% | "
              f"custom dice={summ_b['mean_dice']:.4f} miss={summ_b['complete_miss_rate']*100:.1f}%")
        torch.cuda.empty_cache()
YOLO_TEST = pd.DataFrame(yolo_test_rows)
YOLO_TEST.to_csv(OUT_DIR/"yolo_test_per_seed.csv", index=False)
YOLO_TEST

## §10 · Combined headline table (mean ± std over 3 seeds)

In [ ]:
def agg(df, key):
    g = df.groupby(key).agg(mean_dice=("mean_dice","mean"), std_dice=("mean_dice","std"),
                            median_dice=("median_dice","mean"), miss=("complete_miss_rate","mean"),
                            miss_std=("complete_miss_rate","std")).reset_index()
    return g

seg_agg = agg(SEG_TEST, "model"); seg_agg["variant"] = seg_agg["model"]
yolo_agg = agg(YOLO_TEST.assign(mk=YOLO_TEST.model+" ("+YOLO_TEST.path+")"), "mk").rename(columns={"mk":"variant"})
FINAL = pd.concat([seg_agg[["variant","mean_dice","std_dice","median_dice","miss","miss_std"]],
                   yolo_agg[["variant","mean_dice","std_dice","median_dice","miss","miss_std"]]], ignore_index=True)
FINAL["dice"] = FINAL.apply(lambda r: f"{r.mean_dice:.4f} ± {r.std_dice:.4f}", axis=1)
FINAL["miss_%"] = FINAL.apply(lambda r: f"{r.miss*100:.2f} ± {r.miss_std*100:.2f}", axis=1)
FINAL.to_csv(OUT_DIR/"FINAL_RESULTS.csv", index=False)
print("185 test images, 3 seeds\n" + "="*70)
print(FINAL[["variant","dice","median_dice","miss_%"]].to_string(index=False))

## §11 · Fairness across skin tone (ITA) — every model, both YOLO paths

Per-image Dice averaged over seeds first, then stratified by ITA group. Kruskal–Wallis
(omnibus) + Mann–Whitney (Bonferroni over 10 pairs) + the fairness gap (best − worst
group median). A blank mask on dark skin under-documents an injury, so skin tone is a
primary result, not an ablation.

In [ ]:
def per_image_over_seeds(prefix_keys, store):
    frames = [store[k] for k in prefix_keys if k in store]
    if not frames: return None
    stacked = pd.concat(frames)
    return (stacked.groupby("stem", as_index=False)
            .agg({"dice":"mean","recall":"mean","pred_positive_pixels":"mean","gt_positive_pixels":"first"}))

variants = {}
for name in SEGFORMER_MODELS:
    variants[name] = [f"{name}__seed{s}" for s in CFG["seeds"]]
seg_variant_store = seg_per_image
for name in YOLO_MODELS:
    variants[f"{name} (native_argmax)"] = [f"{name}__seed{s}::native_argmax" for s in CFG["seeds"]]
    variants[f"{name} (custom255)"]     = [f"{name}__seed{s}::custom255" for s in CFG["seeds"]]

fair_group, fair_pair, fair_stats = [], [], []
for variant, keys in variants.items():
    store = seg_per_image if variant in SEGFORMER_MODELS else yolo_per_image
    mpi = per_image_over_seeds(keys, store)
    if mpi is None: continue
    out = fairness_analysis(mpi, MAN["test"], variant)
    fair_group.append(out["per_group"]); fair_pair.append(out["pairwise"]); fair_stats.append(out["stats"])
FAIR_GROUP = pd.concat(fair_group, ignore_index=True)
FAIR_STATS = pd.DataFrame(fair_stats)
FAIR_GROUP.to_csv(OUT_DIR/"fairness_per_group.csv", index=False)
pd.concat(fair_pair, ignore_index=True).to_csv(OUT_DIR/"fairness_pairwise.csv", index=False)
FAIR_STATS.to_csv(OUT_DIR/"fairness_stats.csv", index=False)
print(FAIR_STATS.to_string(index=False))

In [ ]:
import matplotlib.pyplot as plt
pivot = FAIR_GROUP.pivot_table(index="skin_tone_category", columns="model", values="miss_rate") * 100
fig, ax = plt.subplots(figsize=(13,5))
pivot.plot.bar(ax=ax, width=0.85, rot=20)
ax.set_ylabel("complete-miss rate (%)"); ax.set_xlabel("")
ax.set_title("Complete misses by skin tone — a blank mask is a missed injury")
ax.legend(fontsize=7, ncol=2); ax.grid(axis="y", alpha=0.3)
plt.tight_layout(); plt.savefig(OUT_DIR/"fairness_miss.png", dpi=140, bbox_inches="tight"); plt.show()

## ⏱ Speed / inference latency — 640 tensor on GPU → mask on GPU

Same 640 dataloader as training and test scoring (`make_loader` over the 640 cache):
stage every test image on the GPU **once**, so disk read / decode / resize / host→GPU
copy are never inside the timer — they are identical for all five models and dominated
by I/O, so timing them would drown the real architectural differences. We time only
**640-tensor-on-GPU → mask-on-GPU**, per image, with a `cuda.synchronize()` on both
sides (CUDA is asynchronous; without it we'd measure how long it takes to *queue* the
work, not do it). Speed is a property of the architecture, not the seed → seed 0 only.

SegFormer is timed with the library's `benchmark_speed` (its module emits `[B,1,640,640]`
logits). Native YOLO returns a detection tuple, so its raw torch module is timed the same
way (per-image, double-synchronized) minus the mask threshold — raw architecture latency,
directly comparable to the SegFormer numbers on the same staged /255 640 images.

In [ ]:
# -- Stage the 185 test images on the GPU once, via the SAME 640 dataloader ----------
#    The loader returns /255 tensors, which is exactly what both SegFormer (it applies
#    ImageNet norm internally) and the native-YOLO raw module (_probs_640 feeds /255)
#    expect, so one staged batch is valid input for every model.
BENCH_REPEATS = CFG.get("bench_repeats", 3)
BENCH_WARMUP  = CFG.get("bench_warmup", 10)
bench_seed    = CFG["seeds"][0]          # speed is architectural, not per-seed

_bench_loader = make_loader(MAN640["test"], CACHE640, CFG["img_size"], 8, False, CFG["workers"])
GPU_IMAGES = torch.cat([x for x, _, _ in _bench_loader]).to(DEVICE)
print("staged", tuple(GPU_IMAGES.shape),
      f"= {GPU_IMAGES.element_size()*GPU_IMAGES.nelement()/1e9:.2f} GB")

bench_rows = []

# -- SegFormer: trained nn.Module emits [B,1,640,640] logits -> library benchmark_speed --
for name, spec in SEGFORMER_MODELS.items():
    rd = RUNS_DIR / f"{name}__seed{bench_seed}"
    if not (rd / "best.pt").exists():
        continue
    cut = json.loads((rd / "operating_point.json").read_text())["cut"]
    model = build_model(spec["arch"], spec["size"], PATHS).to(DEVICE)
    model.load_state_dict(torch.load(str(rd / "best.pt"), map_location=DEVICE, weights_only=True))
    torch.cuda.reset_peak_memory_stats(DEVICE)
    b = benchmark_speed(model, GPU_IMAGES, DEVICE, cut, BENCH_REPEATS, BENCH_WARMUP)
    b.update({"model": name, "path": "segformer",
              "params_M": count_params(model) / 1e6,
              "peak_activation_MB": torch.cuda.max_memory_allocated(DEVICE) / 1e6})
    bench_rows.append(b)
    print(f"  {name:<26} {b['median_ms']:6.2f} ms  {b['fps']:6.1f} FPS  "
          f"p95={b['p95_ms']:.2f}ms  {b['params_M']:.2f}M")
    del model; torch.cuda.empty_cache()

# -- YOLO: native Ultralytics -> raw torch module, same staged tensor, raw forward only --
#    benchmark_speed's sigmoid+threshold assumes SegFormer's [B,1,H,W] logits; YOLO's raw
#    module returns a detection tuple, so we replicate the exact timing method (per-image,
#    double-synchronized, same repeats/warmup) but time only the forward pass.
@torch.no_grad()
def _benchmark_yolo_forward(model, images, device, repeats, warmup):
    model.eval()
    for _ in range(warmup):
        _ = model(images[:1])
    torch.cuda.synchronize()
    times = []
    for _ in range(repeats):
        for i in range(len(images)):
            x = images[i:i + 1]
            torch.cuda.synchronize(); t0 = time.perf_counter()
            _ = model(x)
            torch.cuda.synchronize(); times.append((time.perf_counter() - t0) * 1000)
    arr = np.array(times)
    return {"median_ms": float(np.median(arr)), "mean_ms": float(arr.mean()),
            "p95_ms": float(np.percentile(arr, 95)), "fps": float(1000.0 / np.median(arr)),
            "n_timed": len(arr)}

for name in YOLO_MODELS:
    best = RUNS_DIR / f"{name}__seed{bench_seed}" / "ultralytics_runs" / "train" / "weights" / "best.pt"
    if not best.exists():
        continue
    model = yn._raw_module(best, DEVICE)          # copy.deepcopy(YOLO(best).model).eval()
    torch.cuda.reset_peak_memory_stats(DEVICE)
    b = _benchmark_yolo_forward(model, GPU_IMAGES, DEVICE, BENCH_REPEATS, BENCH_WARMUP)
    b.update({"model": name, "path": "yolo_native_raw_forward",
              "params_M": sum(p.numel() for p in model.parameters()) / 1e6,
              "peak_activation_MB": torch.cuda.max_memory_allocated(DEVICE) / 1e6})
    bench_rows.append(b)
    print(f"  {name:<26} {b['median_ms']:6.2f} ms  {b['fps']:6.1f} FPS  "
          f"p95={b['p95_ms']:.2f}ms  {b['params_M']:.2f}M")
    del model; torch.cuda.empty_cache()

BENCH = pd.DataFrame(bench_rows)
BENCH.to_csv(OUT_DIR / "benchmark_640.csv", index=False)
del GPU_IMAGES; torch.cuda.empty_cache()
BENCH


## §12 · Annotation ceiling & saved outputs

In [ ]:
IL = pd.read_csv(WORK / "interlabeler_agreement_640.csv")
human = pd.DataFrame([{"comparison": c.replace("_"," ↔ "), "mean_dice": IL[c].mean()}
    for c in ["paul_vs_gbarimah","paul_vs_erik","gbarimah_vs_erik","paul_vs_majority","gbarimah_vs_majority","erik_vs_majority"]
    if c in IL.columns])
print("HUMAN vs HUMAN (same 185 test images):")
print(human.to_string(index=False))
print("\nModel spread sits inside the human spread -> lead with complete-miss rate, not Dice.")
print("\nAll outputs ->", OUT_DIR)
for f in sorted(OUT_DIR.glob("*")): print("   ", f.name)

### How to read this notebook's results

1. **SegFormer** is the healthy, reproducible pipeline — its numbers are the paper's backbone.
2. **YOLO native argmax** is YOLO's *best case* (~0.83 median); **YOLO custom /255** is the
   *same-geometry-as-SegFormer* number (lower). Report both and be explicit which is which.
3. **Miss rate is the honest axis** — it separates the models by more than label noise, and
   it's what matters for injury documentation. Even at its best case, YOLO blanks several %.
4. **Nothing here beats the annotation ceiling** — the model spread is inside the human spread.

## ⭐ Best seed (selected on val) — saved to a separate folder

For every model, pick the seed with the highest **validation** Dice, then report **that seed's
test** result. Selection uses the val set only. SegFormer is scored at its val-fitted cut; YOLO
uses native argmax. Outputs go to `results_final/best_seed_val_selected/` so the per-seed
results above are **not overwritten**.

In [ ]:
BEST_DIR = OUT_DIR / "best_seed_val_selected"
BEST_DIR.mkdir(parents=True, exist_ok=True)
best_rows = []

# ── SegFormer: val + test scored at the val-fitted cut, per seed ──────────────
for name, spec in SEGFORMER_MODELS.items():
    seed_rows, tpi_by_seed = [], {}
    for seed in CFG["seeds"]:
        rd = RUNS_DIR / f"{name}__seed{seed}"
        if not (rd / "best.pt").exists() or not (rd / "operating_point.json").exists():
            continue
        model = build_model(spec["arch"], spec["size"], PATHS).to(DEVICE)
        model.load_state_dict(torch.load(str(rd / "best.pt"), map_location=DEVICE, weights_only=True))
        cut = json.loads((rd / "operating_point.json").read_text())["cut"]
        _,   vs = evaluate_at_cut(model, make_loader(MAN640["val"],  CACHE640, CFG["img_size"], 8, False, CFG["workers"], seed), DEVICE, cut, CFG["amp"])
        tpi, ts = evaluate_at_cut(model, make_loader(MAN640["test"], CACHE640, CFG["img_size"], 8, False, CFG["workers"], seed), DEVICE, cut, CFG["amp"])
        tpi_by_seed[seed] = tpi
        seed_rows.append({"model": name, "path": "segformer", "seed": seed, "cut": cut,
                          "val_mean_dice": vs["mean_dice"],
                          "test_mean_dice": ts["mean_dice"], "test_median_dice": ts["median_dice"],
                          "test_complete_miss_rate": ts["complete_miss_rate"]})
        del model; torch.cuda.empty_cache()
    if not seed_rows:
        continue
    df = pd.DataFrame(seed_rows); b = df.loc[df["val_mean_dice"].idxmax()]; bs = int(b["seed"])
    best_rows.append(b)
    tpi_by_seed[bs].to_csv(BEST_DIR / f"{name}_best_seed{bs}_test_per_image.csv", index=False)
    print(f"{name:<26} best-val seed {bs} | val {b['val_mean_dice']:.4f} -> "
          f"test {b['test_mean_dice']:.4f} (median {b['test_median_dice']:.4f}, "
          f"miss {b['test_complete_miss_rate']*100:.2f}%)")

# ── YOLO: native argmax on val + test, per seed ──────────────────────────────
for name in YOLO_MODELS:
    seed_rows, tpi_by_seed = [], {}
    for seed in CFG["seeds"]:
        best = RUNS_DIR / f"{name}__seed{seed}" / "ultralytics_runs" / "train" / "weights" / "best.pt"
        if not best.exists():
            continue
        _,   vs = yn.predict_native_argmax(best, MAN["val"],  WORK, CFG["img_size"])
        tpi, ts = yn.predict_native_argmax(best, MAN["test"], WORK, CFG["img_size"])
        tpi_by_seed[seed] = tpi
        seed_rows.append({"model": name, "path": "native_argmax", "seed": seed, "cut": float("nan"),
                          "val_mean_dice": vs["mean_dice"],
                          "test_mean_dice": ts["mean_dice"], "test_median_dice": ts["median_dice"],
                          "test_complete_miss_rate": ts["complete_miss_rate"]})
    if not seed_rows:
        continue
    df = pd.DataFrame(seed_rows); b = df.loc[df["val_mean_dice"].idxmax()]; bs = int(b["seed"])
    best_rows.append(b)
    tpi_by_seed[bs].to_csv(BEST_DIR / f"{name}_best_seed{bs}_test_per_image.csv", index=False)
    print(f"{name:<26} best-val seed {bs} | val {b['val_mean_dice']:.4f} -> "
          f"test {b['test_mean_dice']:.4f} (median {b['test_median_dice']:.4f}, "
          f"miss {b['test_complete_miss_rate']*100:.2f}%)")

BEST_SEED_TABLE = pd.DataFrame(best_rows).reset_index(drop=True)
BEST_SEED_TABLE.to_csv(BEST_DIR / "best_seed_val_selected_results.csv", index=False)
print(f"\nsaved -> {BEST_DIR}")
BEST_SEED_TABLE
